In [1]:

# ============================================================
# NOTEBOOK 12 — FINAL RESEARCH EVIDENCE CONSOLIDATION
# CELL 01 — SOURCE AND EVIDENCE INTEGRITY
# ============================================================

from google.colab import drive
from pathlib import Path
import hashlib
import json

drive.mount("/content/drive")

print("=" * 72)
print("CELL 01 — SOURCE AND EVIDENCE INTEGRITY")
print("=" * 72)

PROJECT_ROOT = Path("/content/drive/MyDrive/MSc_Project")

NSL12_CIC_DIR = PROJECT_ROOT / "results/RF_STATS_CIC"

NSL12_NSL09_DIR = (
    PROJECT_ROOT / "results/NSL_KDD_PREPROCESSING"
)

NSL12_NSL10_DIR = (
    PROJECT_ROOT / "results/NSL_KDD_RF_REPLICATION"
)

NSL12_NSL11_DIR = (
    PROJECT_ROOT / "results/CROSS_BENCHMARK_ROBUSTNESS"
)

NSL12_RESULTS_DIR = (
    PROJECT_ROOT / "results/FINAL_RESEARCH_EVIDENCE"
)

NSL12_COMPLETION_PATHS = {
    "Notebook 08":
        NSL12_CIC_DIR / "RF08_NOTEBOOK_COMPLETE.json",
    "Notebook 09":
        NSL12_NSL09_DIR / "NSL09_NOTEBOOK_COMPLETE.json",
    "Notebook 10":
        NSL12_NSL10_DIR / "NSL10_NOTEBOOK_COMPLETE.json",
    "Notebook 11":
        NSL12_NSL11_DIR / "NSL11_NOTEBOOK_COMPLETE.json"
}

NSL12_MANIFEST_PATHS = {
    "NSL09":
        NSL12_NSL09_DIR / "NSL09_manifest.json",
    "NSL10":
        NSL12_NSL10_DIR / "NSL10_manifest.json",
    "NSL11 exports":
        NSL12_NSL11_DIR / "NSL11_export_manifest.json",
    "NSL11 final":
        NSL12_NSL11_DIR / "NSL11_final_manifest.json"
}

def ns12_load_json(path):

    path = Path(path)

    if not path.is_file():
        raise FileNotFoundError(path)

    return json.loads(
        path.read_text(encoding="utf-8")
    )

def ns12_sha256(path):

    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:

        for chunk in iter(
            lambda: handle.read(8 * 1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()

# ------------------------------------------------------------
# 1. Load verified completion records
# ------------------------------------------------------------

NSL12_COMPLETIONS = {
    name: ns12_load_json(path)
    for name, path in NSL12_COMPLETION_PATHS.items()
}

for name, record in NSL12_COMPLETIONS.items():

    if record.get("status") != "COMPLETE":
        raise RuntimeError(
            f"{name}: completion status mismatch."
        )

print("\nPREVIOUS NOTEBOOK COMPLETION")

for name in NSL12_COMPLETIONS:
    print(f"{name}: COMPLETE")

# ------------------------------------------------------------
# 2. Load previous reproducibility manifests
# ------------------------------------------------------------

NSL12_MANIFESTS = {
    name: ns12_load_json(path)
    for name, path in NSL12_MANIFEST_PATHS.items()
}

nsl09 = NSL12_MANIFESTS["NSL09"]
nsl10 = NSL12_MANIFESTS["NSL10"]
nsl11_exports = NSL12_MANIFESTS["NSL11 exports"]
nsl11_final = NSL12_MANIFESTS["NSL11 final"]

expected_preprocessing_fingerprint = (
    "f321a798f07f753bf0de9455e423175590fd1b000259d3d698a83132aa4f5364"
)

if (
    nsl09["combined_artefact_fingerprint"]
    != expected_preprocessing_fingerprint
):
    raise RuntimeError(
        "NSL preprocessing fingerprint mismatch."
    )

if (
    nsl10["source_preprocessing_fingerprint"]
    != expected_preprocessing_fingerprint
):
    raise RuntimeError(
        "NSL replication source fingerprint mismatch."
    )

nsl11_completion = NSL12_COMPLETIONS["Notebook 11"]

if (
    ns12_sha256(NSL12_MANIFEST_PATHS["NSL11 final"])
    != nsl11_completion["final_manifest_sha256"]
):
    raise RuntimeError(
        "Notebook 11 final manifest fingerprint mismatch."
    )

if (
    ns12_sha256(NSL12_MANIFEST_PATHS["NSL11 exports"])
    != nsl11_completion["export_manifest_sha256"]
):
    raise RuntimeError(
        "Notebook 11 export manifest fingerprint mismatch."
    )

# ------------------------------------------------------------
# 3. Verify CIC archived statistical results
# ------------------------------------------------------------

cic_hashes = NSL12_COMPLETIONS[
    "Notebook 08"
]["output_file_hashes"]

if len(cic_hashes) != 17:
    raise RuntimeError(
        "Expected 17 CIC statistical result files."
    )

for filename, expected_hash in cic_hashes.items():

    path = NSL12_CIC_DIR / filename

    if not path.is_file():
        raise FileNotFoundError(path)

    if ns12_sha256(path) != expected_hash:
        raise RuntimeError(
            f"CIC result fingerprint mismatch: {filename}"
        )

# ------------------------------------------------------------
# 4. Verify NSL replication result files
# ------------------------------------------------------------

nsl10_hashes = nsl10["result_file_fingerprints"]

if len(nsl10_hashes) != 22:
    raise RuntimeError(
        "Expected 22 NSL replication result files."
    )

for relative_path, expected_hash in nsl10_hashes.items():

    path = PROJECT_ROOT / relative_path

    if not path.is_file():
        raise FileNotFoundError(path)

    if ns12_sha256(path) != expected_hash:
        raise RuntimeError(
            f"NSL result fingerprint mismatch: {relative_path}"
        )

# ------------------------------------------------------------
# 5. Verify cross-benchmark evidence
# ------------------------------------------------------------

nsl11_hashes = nsl11_exports["evidence_file_sha256"]

if len(nsl11_hashes) != 18:
    raise RuntimeError(
        "Expected 18 cross-benchmark evidence files."
    )

for filename, expected_hash in nsl11_hashes.items():

    path = NSL12_NSL11_DIR / filename

    if not path.is_file():
        raise FileNotFoundError(path)

    if ns12_sha256(path) != expected_hash:
        raise RuntimeError(
            f"Cross-benchmark evidence mismatch: {filename}"
        )

if (
    nsl11_final["evidence_exports"]["file_sha256"]
    != nsl11_hashes
):
    raise RuntimeError(
        "Notebook 11 evidence manifests disagree."
    )

# ------------------------------------------------------------
# 6. Verify referenced source files and model locations
# ------------------------------------------------------------

source_references = nsl11_final[
    "source_file_fingerprints"
]

if len(source_references) != 14:
    raise RuntimeError(
        "Expected 14 source reference files."
    )

for name, record in source_references.items():

    path = PROJECT_ROOT / record["relative_path"]

    if not path.is_file():
        raise FileNotFoundError(path)

    if ns12_sha256(path) != record["sha256"]:
        raise RuntimeError(
            f"Source reference fingerprint mismatch: {name}"
        )

NSL12_MODEL_PATHS = {}

for model_name, relative_path in (
    nsl11_final["model_references"].items()
):

    path = PROJECT_ROOT / relative_path

    if not (path / "saved_model.pb").is_file():
        raise RuntimeError(
            f"Missing SavedModel: {model_name}"
        )

    NSL12_MODEL_PATHS[model_name] = path

expected_models = {
    "RF-B-CIC",
    "RF-E-CIC",
    "RF-B-NSL",
    "RF-H2-NSL"
}

if set(NSL12_MODEL_PATHS) != expected_models:
    raise RuntimeError(
        "Unexpected model reference set."
    )

# ------------------------------------------------------------
# 7. Verify Notebook 11 methodological controls
# ------------------------------------------------------------

controls = nsl11_final["methodological_controls"]

for key in [
    "direct_cic_to_nsl_model_transfer",
    "new_hyperparameter_search",
    "new_resampling",
    "new_model_training",
    "source_experiments_modified"
]:

    if controls[key] is not False:
        raise RuntimeError(
            f"Unexpected experimental operation: {key}"
        )

NSL12_SOURCE_INTEGRITY_VERIFIED = True

print("\nARCHIVED EVIDENCE VERIFICATION")
print("CIC statistical result files:", len(cic_hashes))
print("NSL replication result files:", len(nsl10_hashes))
print("Cross-benchmark evidence files:", len(nsl11_hashes))
print("Source reference files:", len(source_references))
print("Fingerprint verification: PASS")
print("NSL preprocessing fingerprint: PASS")
print("Notebook 11 completion manifest: PASS")

print("\nMODEL REFERENCES")

for model_name in NSL12_MODEL_PATHS:
    print(f"{model_name}: VERIFIED")

print("\nRESEARCH INTEGRITY")
print("Original model configurations: PRESERVED")
print("Existing models modified: NO")
print("Source experiments modified: NO")
print("Additional experiments performed: NO")

print("\nCELL 01: PASSED")
print("Final evidence sources: VERIFIED")
print("Model training performed: NO")
print("=" * 72)


Mounted at /content/drive
CELL 01 — SOURCE AND EVIDENCE INTEGRITY

PREVIOUS NOTEBOOK COMPLETION
Notebook 08: COMPLETE
Notebook 09: COMPLETE
Notebook 10: COMPLETE
Notebook 11: COMPLETE

ARCHIVED EVIDENCE VERIFICATION
CIC statistical result files: 17
NSL replication result files: 22
Cross-benchmark evidence files: 18
Source reference files: 14
Fingerprint verification: PASS
NSL preprocessing fingerprint: PASS
Notebook 11 completion manifest: PASS

MODEL REFERENCES
RF-B-CIC: VERIFIED
RF-B-NSL: VERIFIED
RF-E-CIC: VERIFIED
RF-H2-NSL: VERIFIED

RESEARCH INTEGRITY
Original model configurations: PRESERVED
Existing models modified: NO
Source experiments modified: NO
Additional experiments performed: NO

CELL 01: PASSED
Final evidence sources: VERIFIED
Model training performed: NO


In [2]:

# ============================================================
# NOTEBOOK 12 — FINAL RESEARCH EVIDENCE CONSOLIDATION
# CELL 02 — FINAL MODEL-RESULTS REGISTER
# ============================================================

import numpy as np
import pandas as pd

print("=" * 72)
print("CELL 02 — FINAL MODEL-RESULTS REGISTER")
print("=" * 72)

if not globals().get(
    "NSL12_SOURCE_INTEGRITY_VERIFIED", False
):
    raise RuntimeError("Cell 01 must pass first.")

NSL12_MODEL_REGISTER_VERIFIED = False

# ------------------------------------------------------------
# 1. Reload final independently verified model metrics
# ------------------------------------------------------------

model_metrics_path = (
    NSL12_NSL11_DIR / "NSL11_model_test_metrics.csv"
)

effects_path = (
    NSL12_NSL11_DIR / "NSL11_within_benchmark_effects.csv"
)

NSL12_MODEL_METRICS = pd.read_csv(model_metrics_path)
NSL12_SOURCE_EFFECTS = pd.read_csv(effects_path)

expected_pairs = {
    ("CIC-IDS2017", "RF-B-CIC"),
    ("CIC-IDS2017", "RF-E-CIC"),
    ("NSL-KDD", "RF-B-NSL"),
    ("NSL-KDD", "RF-H2-NSL")
}

observed_pairs = set(
    zip(
        NSL12_MODEL_METRICS["Benchmark"],
        NSL12_MODEL_METRICS["Model"]
    )
)

if (
    len(NSL12_MODEL_METRICS) != 4
    or observed_pairs != expected_pairs
):
    raise RuntimeError(
        "Final model identifiers are incomplete."
    )

metric_names = [
    "Accuracy",
    "Macro Precision",
    "Macro Recall",
    "Macro F1",
    "Weighted F1",
    "Balanced Accuracy",
    "MCC"
]

for metric in metric_names:

    if metric not in NSL12_MODEL_METRICS.columns:
        raise RuntimeError(
            f"Missing model metric: {metric}"
        )

    values = NSL12_MODEL_METRICS[
        metric
    ].to_numpy(dtype=float)

    if not np.isfinite(values).all():
        raise RuntimeError(
            f"Non-finite model metric: {metric}"
        )

    if np.any(values < -1) or np.any(values > 1):
        raise RuntimeError(
            f"Metric outside valid bounds: {metric}"
        )

# ------------------------------------------------------------
# 2. Verify benchmark-specific test configurations
# ------------------------------------------------------------

expected_benchmarks = {
    "CIC-IDS2017": {
        "Classes": 15,
        "Test Records": 378119
    },
    "NSL-KDD": {
        "Classes": 5,
        "Test Records": 22544
    }
}

for benchmark, specification in expected_benchmarks.items():

    subset = NSL12_MODEL_METRICS.loc[
        NSL12_MODEL_METRICS["Benchmark"] == benchmark
    ]

    if len(subset) != 2:
        raise RuntimeError(
            f"{benchmark}: expected two models."
        )

    for column, expected in specification.items():

        if not (
            subset[column].to_numpy(dtype=int)
            == expected
        ).all():
            raise RuntimeError(
                f"{benchmark}: {column} mismatch."
            )

# ------------------------------------------------------------
# 3. Verify known result anchors
# ------------------------------------------------------------

indexed_metrics = NSL12_MODEL_METRICS.set_index(
    "Model"
)

expected_macro_f1 = {
    "RF-B-CIC": 0.893851,
    "RF-E-CIC": 0.912553,
    "RF-B-NSL": 0.504649,
    "RF-H2-NSL": 0.504649
}

for model_name, expected in expected_macro_f1.items():

    actual = float(
        indexed_metrics.loc[model_name, "Macro F1"]
    )

    if not np.isclose(
        actual, expected,
        atol=1e-6,
        rtol=0
    ):
        raise RuntimeError(
            f"{model_name}: Macro F1 mismatch."
        )

# ------------------------------------------------------------
# 4. Construct final model register
# ------------------------------------------------------------

NSL12_MODEL_DESCRIPTIONS = {
    "RF-B-CIC": {
        "Experimental Role": "Baseline",
        "Intervention": "Conventional Random Forest",
        "Training Source": "CIC-IDS2017"
    },
    "RF-E-CIC": {
        "Experimental Role": "Enhanced",
        "Intervention": (
            "Frozen enhanced configuration with "
            "training-only SMOTE"
        ),
        "Training Source": "CIC-IDS2017"
    },
    "RF-B-NSL": {
        "Experimental Role": "Baseline",
        "Intervention": "100-tree Random Forest",
        "Training Source": "NSL-KDD development train"
    },
    "RF-H2-NSL": {
        "Experimental Role": "Frozen configuration",
        "Intervention": (
            "100-tree Random Forest; "
            "maximum depth 30"
        ),
        "Training Source": "NSL-KDD development train"
    }
}

register_records = []

for _, row in NSL12_MODEL_METRICS.iterrows():

    model_name = row["Model"]

    description = NSL12_MODEL_DESCRIPTIONS[
        model_name
    ]

    record = {
        "Benchmark": row["Benchmark"],
        "Model": model_name,
        "Experimental Role": description[
            "Experimental Role"
        ],
        "Training Source": description[
            "Training Source"
        ],
        "Intervention": description[
            "Intervention"
        ],
        "Test Records": int(row["Test Records"]),
        "Classes": int(row["Classes"])
    }

    for metric in metric_names:
        record[metric] = float(row[metric])

    register_records.append(record)

NSL12_MODEL_REGISTER = pd.DataFrame(
    register_records
)

if len(NSL12_MODEL_REGISTER) != 4:
    raise RuntimeError(
        "Unexpected final model register length."
    )

# ------------------------------------------------------------
# 5. Recalculate within-benchmark changes
# ------------------------------------------------------------

comparison_specifications = {
    "CIC-IDS2017": (
        "RF-B-CIC",
        "RF-E-CIC"
    ),
    "NSL-KDD": (
        "RF-B-NSL",
        "RF-H2-NSL"
    )
}

effect_records = []

for benchmark, (baseline, comparison) in (
    comparison_specifications.items()
):

    for metric in metric_names:

        baseline_value = float(
            indexed_metrics.loc[baseline, metric]
        )

        comparison_value = float(
            indexed_metrics.loc[comparison, metric]
        )

        effect_records.append({
            "Benchmark": benchmark,
            "Baseline Model": baseline,
            "Comparison Model": comparison,
            "Metric": metric,
            "Baseline Value": baseline_value,
            "Comparison Value": comparison_value,
            "Change": comparison_value - baseline_value
        })

NSL12_FINAL_EFFECTS = pd.DataFrame(effect_records)

if len(NSL12_FINAL_EFFECTS) != 14:
    raise RuntimeError(
        "Expected 14 metric-effect records."
    )

# Reconcile against archived Notebook 11 effects.

for _, row in NSL12_FINAL_EFFECTS.iterrows():

    subset = NSL12_SOURCE_EFFECTS.loc[
        (
            NSL12_SOURCE_EFFECTS["Benchmark"]
            == row["Benchmark"]
        )
        &
        (
            NSL12_SOURCE_EFFECTS["Metric"]
            == row["Metric"]
        )
    ]

    if len(subset) != 1:
        raise RuntimeError(
            "Archived effect record unavailable."
        )

    archived_change = float(
        subset.iloc[0][
            "Within-Benchmark Change"
        ]
    )

    if not np.isclose(
        float(row["Change"]),
        archived_change,
        atol=3e-9,
        rtol=0
    ):
        raise RuntimeError(
            f"Effect reconciliation failed: "
            f"{row['Benchmark']}, {row['Metric']}"
        )

# ------------------------------------------------------------
# 6. Verify important effect estimates
# ------------------------------------------------------------

def ns12_get_effect(benchmark, metric):

    selected = NSL12_FINAL_EFFECTS.loc[
        (
            NSL12_FINAL_EFFECTS["Benchmark"]
            == benchmark
        )
        &
        (
            NSL12_FINAL_EFFECTS["Metric"]
            == metric
        )
    ]

    if len(selected) != 1:
        raise RuntimeError(
            "Expected one benchmark-specific effect."
        )

    return float(selected.iloc[0]["Change"])

cic_f1_change = ns12_get_effect(
    "CIC-IDS2017", "Macro F1"
)

cic_recall_change = ns12_get_effect(
    "CIC-IDS2017", "Macro Recall"
)

nsl_f1_change = ns12_get_effect(
    "NSL-KDD", "Macro F1"
)

if not np.isclose(
    cic_f1_change,
    0.018702,
    atol=2e-6,
    rtol=0
):
    raise RuntimeError(
        "CIC Macro F1 effect mismatch."
    )

if not np.isclose(
    cic_recall_change,
    0.048248,
    atol=2e-6,
    rtol=0
):
    raise RuntimeError(
        "CIC Macro Recall effect mismatch."
    )

if not np.isclose(
    nsl_f1_change,
    0.0,
    atol=1e-9,
    rtol=0
):
    raise RuntimeError(
        "NSL Macro F1 effect mismatch."
    )

NSL12_MODEL_REGISTER_VERIFIED = True

print("\nFINAL MODEL RESULTS")

print(
    NSL12_MODEL_REGISTER[
        [
            "Benchmark",
            "Model",
            "Test Records",
            "Classes",
            "Accuracy",
            "Macro Precision",
            "Macro Recall",
            "Macro F1",
            "MCC"
        ]
    ].to_string(
        index=False,
        float_format=lambda value: f"{value:.6f}"
    )
)

print("\nWITHIN-BENCHMARK EFFECTS")

print(
    NSL12_FINAL_EFFECTS[
        [
            "Benchmark",
            "Metric",
            "Baseline Value",
            "Comparison Value",
            "Change"
        ]
    ].to_string(
        index=False,
        float_format=lambda value: f"{value:+.6f}"
    )
)

print("\nMODEL REGISTER VERIFICATION")
print("Verified model records:", len(NSL12_MODEL_REGISTER))
print("Benchmark-specific effects:", len(NSL12_FINAL_EFFECTS))
print("CIC Macro F1 change:", f"{cic_f1_change:+.6f}")
print("CIC Macro Recall change:", f"{cic_recall_change:+.6f}")
print("NSL Macro F1 change:", f"{nsl_f1_change:+.6f}")
print("Archived effect reconciliation: PASS")
print("Cross-benchmark raw metric ranking: EXCLUDED")

print("\nCELL 02: PASSED")
print("Final experimental model register: VERIFIED")
print("Model training performed: NO")
print("=" * 72)


CELL 02 — FINAL MODEL-RESULTS REGISTER

FINAL MODEL RESULTS
  Benchmark     Model  Test Records  Classes  Accuracy  Macro Precision  Macro Recall  Macro F1      MCC
CIC-IDS2017  RF-B-CIC        378119       15  0.998606         0.935485      0.879297  0.893851 0.995364
CIC-IDS2017  RF-E-CIC        378119       15  0.998633         0.901210      0.927545  0.912553 0.995457
    NSL-KDD  RF-B-NSL         22544        5  0.758384         0.766484      0.493121  0.504649 0.650888
    NSL-KDD RF-H2-NSL         22544        5  0.758384         0.766484      0.493121  0.504649 0.650888

WITHIN-BENCHMARK EFFECTS
  Benchmark            Metric  Baseline Value  Comparison Value    Change
CIC-IDS2017          Accuracy       +0.998606         +0.998633 +0.000026
CIC-IDS2017   Macro Precision       +0.935485         +0.901210 -0.034275
CIC-IDS2017      Macro Recall       +0.879297         +0.927545 +0.048248
CIC-IDS2017          Macro F1       +0.893851         +0.912553 +0.018702
CIC-IDS2017       W

In [3]:

# ============================================================
# NOTEBOOK 12 — FINAL RESEARCH EVIDENCE CONSOLIDATION
# CELL 03 — RESEARCH FINDINGS AND EVIDENCE BOUNDARIES
# ============================================================

import numpy as np
import pandas as pd

print("=" * 72)
print("CELL 03 — RESEARCH FINDINGS AND EVIDENCE BOUNDARIES")
print("=" * 72)

if not globals().get(
    "NSL12_MODEL_REGISTER_VERIFIED", False
):
    raise RuntimeError("Cell 02 must pass first.")

NSL12_FINDINGS_VERIFIED = False

# ------------------------------------------------------------
# 1. Reload verified supporting evidence
# ------------------------------------------------------------

NSL12_INTERPRETATION = ns12_load_json(
    NSL12_NSL11_DIR / "NSL11_interpretation_record.json"
)

NSL12_CLASS_EFFECTS = pd.read_csv(
    NSL12_NSL11_DIR / "NSL11_class_effects.csv"
)

NSL12_NSL_GAPS = pd.read_csv(
    NSL12_NSL11_DIR / "NSL11_nsl_generalisation_gap.csv"
)

NSL12_NOVELTY_COHORTS = pd.read_csv(
    NSL12_NSL11_DIR
    / "NSL11_nsl_novelty_cohort_performance.csv"
)

NSL12_FAMILY_SHIFT = pd.read_csv(
    NSL12_NSL11_DIR
    / "NSL11_nsl_family_distribution_shift.csv"
)

NSL12_NSL_TEST_REPORT = pd.read_csv(
    NSL12_NSL10_DIR
    / "RF_B_NSL/RF_B_NSL_test_classification_report.csv",
    index_col=0
)

NSL12_NSL_AGREEMENT = ns12_load_json(
    NSL12_NSL10_DIR
    / "NSL10_prediction_agreement.json"
)

# ------------------------------------------------------------
# 2. Verify CIC class-level findings
# ------------------------------------------------------------

cic_class_effects = NSL12_CLASS_EFFECTS.loc[
    NSL12_CLASS_EFFECTS["Benchmark"]
    == "CIC-IDS2017"
]

nsl_class_effects = NSL12_CLASS_EFFECTS.loc[
    NSL12_CLASS_EFFECTS["Benchmark"]
    == "NSL-KDD"
]

if len(cic_class_effects) != 15:
    raise RuntimeError("Expected 15 CIC class records.")

if len(nsl_class_effects) != 5:
    raise RuntimeError("Expected five NSL class records.")

cic_improved = int(
    (cic_class_effects["Direction"] == "Improvement").sum()
)

cic_deteriorated = int(
    (
        cic_class_effects["Direction"]
        == "Deterioration"
    ).sum()
)

cic_unchanged = int(
    (
        cic_class_effects["Direction"]
        == "No observed change"
    ).sum()
)

if (
    cic_improved,
    cic_deteriorated,
    cic_unchanged
) != (8, 3, 4):
    raise RuntimeError(
        "CIC class-effect distribution mismatch."
    )

if not (
    nsl_class_effects["Direction"] == "No observed change"
).all():
    raise RuntimeError(
        "NSL model class effects are not identical."
    )

# ------------------------------------------------------------
# 3. Verify paired statistical evidence
# ------------------------------------------------------------

cic_mcnemar_p = float(
    NSL12_INTERPRETATION[
        "cic_ids2017"
    ]["mcnemar_p_value"]
)

if not np.isclose(
    cic_mcnemar_p,
    0.5384046,
    rtol=0,
    atol=1e-6
):
    raise RuntimeError(
        "CIC McNemar p-value mismatch."
    )

if NSL12_INTERPRETATION[
    "cic_ids2017"
]["macro_f1_significance_test_performed"] is not False:
    raise RuntimeError(
        "Macro F1 significance scope mismatch."
    )

# ------------------------------------------------------------
# 4. Verify NSL generalisation gap
# ------------------------------------------------------------

nsl_f1_gap_rows = NSL12_NSL_GAPS.loc[
    (
        NSL12_NSL_GAPS["Model"]
        == "RF-B-NSL"
    )
    &
    (
        NSL12_NSL_GAPS["Metric"]
        == "Macro F1"
    )
]

if len(nsl_f1_gap_rows) != 1:
    raise RuntimeError(
        "NSL Macro F1 generalisation record missing."
    )

nsl_f1_gap_record = nsl_f1_gap_rows.iloc[0]

nsl_validation_f1 = float(
    nsl_f1_gap_record["Validation"]
)

nsl_test_f1 = float(
    nsl_f1_gap_record["Official Test"]
)

nsl_generalisation_gap = float(
    nsl_f1_gap_record["Test Minus Validation"]
)

if not np.isclose(
    nsl_generalisation_gap,
    nsl_test_f1 - nsl_validation_f1,
    rtol=0,
    atol=3e-9
):
    raise RuntimeError(
        "NSL generalisation gap reconciliation failed."
    )

if not np.isclose(
    nsl_generalisation_gap,
    -0.453498,
    rtol=0,
    atol=2e-6
):
    raise RuntimeError(
        "Unexpected NSL Macro F1 generalisation gap."
    )

# ------------------------------------------------------------
# 5. Verify NSL attack-novelty cohorts
# ------------------------------------------------------------

novelty = NSL12_NOVELTY_COHORTS.set_index("Cohort")

expected_cohorts = {
    "Training-seen attack names",
    "Test-only attack names"
}

if set(novelty.index) != expected_cohorts:
    raise RuntimeError(
        "Attack novelty cohorts differ."
    )

seen = novelty.loc["Training-seen attack names"]
unseen = novelty.loc["Test-only attack names"]

seen_support = int(seen["Support"])
seen_correct = int(seen["Correct"])

unseen_support = int(unseen["Support"])
unseen_correct = int(unseen["Correct"])
unseen_errors = int(unseen["Incorrect"])

if (
    seen_support,
    seen_correct,
    unseen_support,
    unseen_correct,
    unseen_errors
) != (
    18794,
    16398,
    3750,
    699,
    3051
):
    raise RuntimeError(
        "NSL attack-novelty evidence mismatch."
    )

total_correct = seen_correct + unseen_correct
total_errors = (
    int(seen["Incorrect"]) + unseen_errors
)

if (total_correct, total_errors) != (17097, 5447):
    raise RuntimeError(
        "NSL official-test totals mismatch."
    )

seen_accuracy = seen_correct / seen_support
unseen_accuracy = unseen_correct / unseen_support

unseen_test_share = unseen_support / 22544
unseen_error_share = unseen_errors / total_errors

# ------------------------------------------------------------
# 6. Verify class distribution and minority recall
# ------------------------------------------------------------

family_shift = NSL12_FAMILY_SHIFT.set_index("Family")

if set(family_shift.index) != {
    "Normal", "DoS", "Probe", "R2L", "U2R"
}:
    raise RuntimeError(
        "Unexpected NSL attack-family distribution."
    )

r2l_training_count = int(
    family_shift.loc["R2L", "Training Count"]
)

r2l_test_count = int(
    family_shift.loc["R2L", "Testing Count"]
)

if (r2l_training_count, r2l_test_count) != (
    995, 2885
):
    raise RuntimeError(
        "R2L family distribution mismatch."
    )

r2l_training_share = float(
    family_shift.loc["R2L", "Training Proportion"]
)

r2l_test_share = float(
    family_shift.loc["R2L", "Test Proportion"]
)

r2l_recall = float(
    NSL12_NSL_TEST_REPORT.loc["R2L", "recall"]
)

u2r_recall = float(
    NSL12_NSL_TEST_REPORT.loc["U2R", "recall"]
)

if not np.isclose(
    r2l_recall, 0.054766,
    atol=1e-6,
    rtol=0
):
    raise RuntimeError(
        "R2L official-test recall mismatch."
    )

if not np.isclose(
    u2r_recall, 0.029851,
    atol=1e-6,
    rtol=0
):
    raise RuntimeError(
        "U2R official-test recall mismatch."
    )

if NSL12_NSL_AGREEMENT[
    "validation_prediction_disagreements"
] != 0:
    raise RuntimeError(
        "NSL validation prediction disagreement."
    )

if NSL12_NSL_AGREEMENT[
    "official_test_prediction_disagreements"
] != 0:
    raise RuntimeError(
        "NSL test prediction disagreement."
    )

# ------------------------------------------------------------
# 7. Construct dissertation evidence register
# ------------------------------------------------------------

NSL12_FINDING_RECORDS = [
    {
        "Finding ID": "F01",
        "Research Dimension": "Class imbalance",
        "Finding": "CIC enhancement improved Macro F1",
        "Numerical Evidence": (
            f"{ns12_get_effect('CIC-IDS2017', 'Macro F1'):+.6f}"
        ),
        "Interpretation": (
            "Observed class-balanced performance improvement "
            "with heterogeneous class-level effects"
        ),
        "Source": (
            "NSL11_within_benchmark_effects.csv"
        )
    },
    {
        "Finding ID": "F02",
        "Research Dimension": "Detection trade-offs",
        "Finding": "CIC Macro Recall increased",
        "Numerical Evidence": (
            f"{ns12_get_effect('CIC-IDS2017', 'Macro Recall'):+.6f}"
        ),
        "Interpretation": (
            "Higher observed recall accompanied by lower "
            "Macro Precision"
        ),
        "Source": (
            "NSL11_within_benchmark_effects.csv"
        )
    },
    {
        "Finding ID": "F03",
        "Research Dimension": "Class-level robustness",
        "Finding": "CIC class-level changes were mixed",
        "Numerical Evidence": (
            f"{cic_improved} improved; "
            f"{cic_deteriorated} deteriorated; "
            f"{cic_unchanged} unchanged"
        ),
        "Interpretation": (
            "The enhancement did not consistently benefit "
            "every attack class"
        ),
        "Source": "NSL11_class_effects.csv"
    },
    {
        "Finding ID": "F04",
        "Research Dimension": "Statistical evaluation",
        "Finding": "CIC overall correctness difference",
        "Numerical Evidence": (
            f"McNemar p={cic_mcnemar_p:.7f}"
        ),
        "Interpretation": (
            "Not statistically significant at alpha 0.05 "
            "for paired overall correctness"
        ),
        "Source": "NSL11_interpretation_record.json"
    },
    {
        "Finding ID": "F05",
        "Research Dimension": "Method replication",
        "Finding": "NSL frozen depth restriction had no prediction effect",
        "Numerical Evidence": (
            "0 validation and 0 official-test disagreements"
        ),
        "Interpretation": (
            "Observed classification predictions are identical; "
            "forest structures are not assumed identical"
        ),
        "Source": "NSL10_prediction_agreement.json"
    },
    {
        "Finding ID": "F06",
        "Research Dimension": "Generalisation",
        "Finding": "NSL Macro F1 deteriorated on official test",
        "Numerical Evidence": (
            f"{nsl_validation_f1:.6f} to "
            f"{nsl_test_f1:.6f}; "
            f"change {nsl_generalisation_gap:+.6f}"
        ),
        "Interpretation": (
            "Substantial development-to-official-test "
            "performance deterioration"
        ),
        "Source": "NSL11_nsl_generalisation_gap.csv"
    },
    {
        "Finding ID": "F07",
        "Research Dimension": "Unseen attack types",
        "Finding": "Test-only attack names were difficult to detect",
        "Numerical Evidence": (
            f"{unseen_support} records; "
            f"{unseen_accuracy:.6f} accuracy"
        ),
        "Interpretation": (
            "Descriptive difficulty with attack names "
            "absent from the training partition"
        ),
        "Source": (
            "NSL11_nsl_novelty_cohort_performance.csv"
        )
    },
    {
        "Finding ID": "F08",
        "Research Dimension": "Error concentration",
        "Finding": "Test-only attacks contributed most errors",
        "Numerical Evidence": (
            f"{unseen_errors}/{total_errors} errors; "
            f"{100 * unseen_error_share:.3f}%"
        ),
        "Interpretation": (
            "Test-only names account for a disproportionate "
            "share of errors"
        ),
        "Source": (
            "NSL11_nsl_novelty_cohort_performance.csv"
        )
    },
    {
        "Finding ID": "F09",
        "Research Dimension": "Dataset shift",
        "Finding": "NSL R2L class prevalence increased",
        "Numerical Evidence": (
            f"{100 * r2l_training_share:.3f}% to "
            f"{100 * r2l_test_share:.3f}%"
        ),
        "Interpretation": (
            "Official training and test class distributions "
            "are materially different"
        ),
        "Source": (
            "NSL11_nsl_family_distribution_shift.csv"
        )
    },
    {
        "Finding ID": "F10",
        "Research Dimension": "Minority detection",
        "Finding": "NSL R2L and U2R recall remained low",
        "Numerical Evidence": (
            f"R2L={r2l_recall:.6f}; "
            f"U2R={u2r_recall:.6f}"
        ),
        "Interpretation": (
            "Important minority-attack detection weaknesses "
            "on the official test partition"
        ),
        "Source": (
            "RF_B_NSL_test_classification_report.csv"
        )
    }
]

NSL12_FINDINGS_REGISTER = pd.DataFrame(
    NSL12_FINDING_RECORDS
)

if len(NSL12_FINDINGS_REGISTER) != 10:
    raise RuntimeError(
        "Expected ten verified research findings."
    )

if NSL12_FINDINGS_REGISTER[
    "Finding ID"
].duplicated().any():
    raise RuntimeError(
        "Duplicate research finding identifier."
    )

# ------------------------------------------------------------
# 8. Define methodological interpretation boundaries
# ------------------------------------------------------------

NSL12_INTERPRETATION_BOUNDARIES = [
    (
        "CIC-IDS2017 and NSL-KDD use different target "
        "label spaces: 15 classes versus five families."
    ),
    (
        "CIC enhancement effects include training-only "
        "SMOTE; the independent NSL experiment evaluates "
        "a frozen maximum-depth configuration."
    ),
    (
        "No fitted CIC model or feature representation "
        "was transferred directly to NSL-KDD."
    ),
    (
        "Identical predictions do not establish identical "
        "Random Forest structures."
    ),
    (
        "The NSL validation-to-test performance decline "
        "cannot be assigned to attack novelty alone."
    ),
    (
        "Training-seen and test-only attack-name cohorts "
        "have different class-family compositions."
    ),
    (
        "CIC McNemar significance applies to paired "
        "overall correctness, not the Macro F1 difference."
    ),
    (
        "Rare-class estimates require caution because "
        "some class supports are very small."
    ),
    (
        "Potential dependence among network flows and "
        "test-set reuse constrain confirmatory claims."
    )
]

if len(NSL12_INTERPRETATION_BOUNDARIES) != 9:
    raise RuntimeError(
        "Interpretation boundary count mismatch."
    )

NSL12_FINDINGS_VERIFIED = True

print("\nPRIMARY RESEARCH FINDINGS")

print(
    NSL12_FINDINGS_REGISTER[
        [
            "Finding ID",
            "Research Dimension",
            "Finding",
            "Numerical Evidence"
        ]
    ].to_string(index=False)
)

print("\nGENERALISATION EVIDENCE")
print("NSL validation Macro F1:", f"{nsl_validation_f1:.6f}")
print("NSL official-test Macro F1:", f"{nsl_test_f1:.6f}")
print("Generalisation gap:", f"{nsl_generalisation_gap:+.6f}")
print("Training-seen attack accuracy:", f"{seen_accuracy:.6f}")
print("Test-only attack accuracy:", f"{unseen_accuracy:.6f}")
print("Test-only attack observations:", unseen_support)
print("Test-only share of errors:", f"{100 * unseen_error_share:.3f}%")
print("R2L official-test recall:", f"{r2l_recall:.6f}")
print("U2R official-test recall:", f"{u2r_recall:.6f}")

print("\nINTERPRETATION BOUNDARIES")
for index, boundary in enumerate(
    NSL12_INTERPRETATION_BOUNDARIES,
    start=1
):
    print(f"{index}. {boundary}")

print("\nFINDINGS INTEGRITY")
print("Research findings:", len(NSL12_FINDINGS_REGISTER))
print("Interpretation boundaries:", len(NSL12_INTERPRETATION_BOUNDARIES))
print("CIC class-level effects: VERIFIED")
print("CIC statistical scope: VERIFIED")
print("NSL model agreement: VERIFIED")
print("NSL generalisation gap: VERIFIED")
print("Attack-novelty cohort totals: VERIFIED")
print("Minority-class recall: VERIFIED")
print("Direct cross-dataset model transfer claim: EXCLUDED")

print("\nCELL 03: PASSED")
print("Final research findings register: VERIFIED")
print("Model training performed: NO")
print("=" * 72)


CELL 03 — RESEARCH FINDINGS AND EVIDENCE BOUNDARIES

PRIMARY RESEARCH FINDINGS
Finding ID     Research Dimension                                               Finding                             Numerical Evidence
       F01        Class imbalance                     CIC enhancement improved Macro F1                                      +0.018702
       F02   Detection trade-offs                            CIC Macro Recall increased                                      +0.048248
       F03 Class-level robustness                    CIC class-level changes were mixed        8 improved; 3 deteriorated; 4 unchanged
       F04 Statistical evaluation                    CIC overall correctness difference                            McNemar p=0.5384046
       F05     Method replication NSL frozen depth restriction had no prediction effect 0 validation and 0 official-test disagreements
       F06         Generalisation            NSL Macro F1 deteriorated on official test         0.958147 to 0.5

In [4]:
# ======================================================================
# CELL 04 — FINAL DISSERTATION RESULTS TABLES
# ======================================================================

from pathlib import Path
import pandas as pd

print("=" * 72)
print("CELL 04 — FINAL DISSERTATION RESULTS TABLES")
print("=" * 72)

# ----------------------------------------------------------------------
# 1. Verify notebook prerequisites
# ----------------------------------------------------------------------

required_objects = [
    "PROJECT_ROOT",
    "NSL12_RESULTS_DIR",
    "NSL12_NSL11_DIR",
    "NSL12_MODEL_REGISTER",
    "NSL12_FINAL_EFFECTS",
    "NSL12_FINDINGS_REGISTER",
    "NSL12_INTERPRETATION_BOUNDARIES"
]

missing = [
    name for name in required_objects
    if name not in globals()
]

if missing:
    raise RuntimeError(
        "Missing Notebook 12 prerequisites: "
        + ", ".join(missing)
        + ". Run Cells 01–03 first."
    )

NSL12_OUTPUT_DIR = Path(NSL12_RESULTS_DIR)
NSL12_SOURCE_DIR = Path(NSL12_NSL11_DIR)

assert NSL12_SOURCE_DIR.is_dir(), (
    "Notebook 11 evidence directory not found."
)

assert len(NSL12_MODEL_REGISTER) == 4
assert len(NSL12_FINAL_EFFECTS) == 14
assert len(NSL12_FINDINGS_REGISTER) == 10
assert len(NSL12_INTERPRETATION_BOUNDARIES) == 9

assert set(NSL12_MODEL_REGISTER["Model"]) == {
    "RF-B-CIC",
    "RF-E-CIC",
    "RF-B-NSL",
    "RF-H2-NSL"
}

# ----------------------------------------------------------------------
# 2. Read archived Notebook 11 tables
# ----------------------------------------------------------------------

source_tables = {
    "class_effects":
        "NSL11_class_effects.csv",

    "nsl_class_generalisation":
        "NSL11_nsl_class_generalisation.csv",

    "nsl_family_shift":
        "NSL11_nsl_family_distribution_shift.csv",

    "nsl_novelty":
        "NSL11_nsl_novelty_cohort_performance.csv",

    "nsl_test_only_attacks":
        "NSL11_nsl_test_only_attack_inventory.csv"
}

archived_tables = {}

for key, filename in source_tables.items():

    source = NSL12_SOURCE_DIR / filename

    if not source.is_file():
        raise FileNotFoundError(source)

    archived_tables[key] = pd.read_csv(source)

expected_source_rows = {
    "class_effects": 20,
    "nsl_class_generalisation": 10,
    "nsl_family_shift": 5,
    "nsl_novelty": 2,
    "nsl_test_only_attacks": 17
}

for key, count in expected_source_rows.items():
    assert len(archived_tables[key]) == count, (
        f"Unexpected source row count: {key}"
    )

# ----------------------------------------------------------------------
# 3. Prepare final dissertation tables
# ----------------------------------------------------------------------

boundaries_df = pd.DataFrame({
    "Boundary ID": [
        f"B{i:02d}" for i in range(
            1,
            len(NSL12_INTERPRETATION_BOUNDARIES) + 1
        )
    ],
    "Interpretation Boundary":
        NSL12_INTERPRETATION_BOUNDARIES
})

NSL12_FINAL_TABLES = {
    "NSL12_final_model_results.csv":
        NSL12_MODEL_REGISTER.copy(),

    "NSL12_within_benchmark_effects.csv":
        NSL12_FINAL_EFFECTS.copy(),

    "NSL12_class_level_effects.csv":
        archived_tables["class_effects"].copy(),

    "NSL12_nsl_class_generalisation.csv":
        archived_tables["nsl_class_generalisation"].copy(),

    "NSL12_nsl_family_distribution_shift.csv":
        archived_tables["nsl_family_shift"].copy(),

    "NSL12_nsl_attack_novelty.csv":
        archived_tables["nsl_novelty"].copy(),

    "NSL12_nsl_test_only_attacks.csv":
        archived_tables["nsl_test_only_attacks"].copy(),

    "NSL12_research_findings.csv":
        NSL12_FINDINGS_REGISTER.copy(),

    "NSL12_interpretation_boundaries.csv":
        boundaries_df
}

# ----------------------------------------------------------------------
# 4. Controlled export — Notebook 12 directory only
# ----------------------------------------------------------------------

NSL12_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def ns12_export_table(frame, filename):

    destination = NSL12_OUTPUT_DIR / filename

    assert destination.resolve().parent == (
        NSL12_OUTPUT_DIR.resolve()
    ), "Export destination is outside Notebook 12."

    if destination.exists():

        existing = pd.read_csv(destination)

        pd.testing.assert_frame_equal(
            existing.reset_index(drop=True),
            frame.reset_index(drop=True),
            check_dtype=False,
            check_exact=False,
            rtol=1e-10,
            atol=1e-12
        )

        status = "VERIFIED EXISTING"

    else:

        frame.to_csv(destination, index=False)
        status = "CREATED"

    return destination, status

NSL12_CELL04_FILES = []

print("\nFINAL RESULTS TABLE EXPORTS")

for filename, frame in NSL12_FINAL_TABLES.items():

    destination, status = ns12_export_table(
        frame,
        filename
    )

    NSL12_CELL04_FILES.append(destination)

    print(
        f"{filename}: {status} "
        f"({len(frame)} rows)"
    )

# ----------------------------------------------------------------------
# 5. Verify exported contents
# ----------------------------------------------------------------------

for filename, original in NSL12_FINAL_TABLES.items():

    saved = pd.read_csv(
        NSL12_OUTPUT_DIR / filename
    )

    assert len(saved) == len(original)
    assert list(saved.columns) == list(original.columns)

print("\nTABLE INTEGRITY")
print("Final tables:", len(NSL12_CELL04_FILES))
print("Model records:", len(NSL12_MODEL_REGISTER))
print("Benchmark effects:", len(NSL12_FINAL_EFFECTS))
print("Class-effect records:", len(archived_tables["class_effects"]))
print("Research findings:", len(NSL12_FINDINGS_REGISTER))
print("Interpretation boundaries:", len(boundaries_df))
print("Saved table verification: PASS")
print("Source experiments modified: NO")

print("\nCELL 04: PASSED")
print("Final dissertation tables: VERIFIED")
print("Model training performed: NO")
print("=" * 72)

CELL 04 — FINAL DISSERTATION RESULTS TABLES

FINAL RESULTS TABLE EXPORTS
NSL12_final_model_results.csv: CREATED (4 rows)
NSL12_within_benchmark_effects.csv: CREATED (14 rows)
NSL12_class_level_effects.csv: CREATED (20 rows)
NSL12_nsl_class_generalisation.csv: CREATED (10 rows)
NSL12_nsl_family_distribution_shift.csv: CREATED (5 rows)
NSL12_nsl_attack_novelty.csv: CREATED (2 rows)
NSL12_nsl_test_only_attacks.csv: CREATED (17 rows)
NSL12_research_findings.csv: CREATED (10 rows)
NSL12_interpretation_boundaries.csv: CREATED (9 rows)

TABLE INTEGRITY
Final tables: 9
Model records: 4
Benchmark effects: 14
Class-effect records: 20
Research findings: 10
Interpretation boundaries: 9
Saved table verification: PASS
Source experiments modified: NO

CELL 04: PASSED
Final dissertation tables: VERIFIED
Model training performed: NO


In [5]:
# ======================================================================
# CELL 05 — FINAL FIGURE AND VISUAL-EVIDENCE INDEX
# ======================================================================

import hashlib
import pandas as pd
from pathlib import Path
from PIL import Image

print("=" * 72)
print("CELL 05 — FINAL FIGURE AND VISUAL-EVIDENCE INDEX")
print("=" * 72)

assert "NSL12_SOURCE_DIR" in globals()
assert "NSL12_OUTPUT_DIR" in globals()
assert "ns12_export_table" in globals()

# ----------------------------------------------------------------------
# 1. Define verified Notebook 11 figures
# ----------------------------------------------------------------------

figure_specs = [
    (
        "FIG-01",
        "NSL11_within_benchmark_metric_effects.png",
        1980,
        1196,
        "Within-benchmark performance effects",
        "Compare enhancement effects separately within CIC-IDS2017 and NSL-KDD"
    ),
    (
        "FIG-02",
        "NSL11_nsl_validation_vs_test_class_f1.png",
        2179,
        1294,
        "NSL-KDD class-level generalisation",
        "Illustrate class-level F1 deterioration between validation and official testing"
    ),
    (
        "FIG-03",
        "NSL11_nsl_attack_family_distribution.png",
        1980,
        1179,
        "NSL-KDD attack-family distribution shift",
        "Illustrate changes in attack-family prevalence between training and testing"
    ),
    (
        "FIG-04",
        "NSL11_nsl_attack_novelty_accuracy.png",
        1780,
        1196,
        "NSL-KDD attack-novelty cohort performance",
        "Compare accuracy for training-seen and test-only attack-name cohorts"
    ),
    (
        "FIG-05",
        "NSL11_nsl_family_novelty_recall.png",
        2180,
        1196,
        "NSL-KDD family-level novelty performance",
        "Examine differences in attack-family recall across novelty cohorts"
    )
]

# ----------------------------------------------------------------------
# 2. Verify each figure
# ----------------------------------------------------------------------

figure_records = []

for (
    figure_id,
    filename,
    expected_width,
    expected_height,
    caption,
    purpose
) in figure_specs:

    figure_path = NSL12_SOURCE_DIR / filename

    if not figure_path.is_file():
        raise FileNotFoundError(figure_path)

    with Image.open(figure_path) as img:

        image_format = img.format
        width, height = img.size

        img.verify()

    assert image_format == "PNG"
    assert width == expected_width
    assert height == expected_height

    sha256 = hashlib.sha256(
        figure_path.read_bytes()
    ).hexdigest()

    figure_records.append({
        "Figure ID": figure_id,
        "Source Notebook": "11_Cross_Benchmark_Robustness_Analysis",
        "Filename": filename,
        "Relative Path": str(
            figure_path.relative_to(Path(PROJECT_ROOT))
        ),
        "Caption": caption,
        "Research Purpose": purpose,
        "Width": width,
        "Height": height,
        "SHA256": sha256,
        "Verification": "PASS"
    })

NSL12_FIGURE_INDEX = pd.DataFrame(figure_records)

# ----------------------------------------------------------------------
# 3. Export verified figure index
# ----------------------------------------------------------------------

figure_index_path, export_status = ns12_export_table(
    NSL12_FIGURE_INDEX,
    "NSL12_final_figure_index.csv"
)

NSL12_CELL05_FILES = [figure_index_path]

print("\nVERIFIED RESEARCH FIGURES")

print(
    NSL12_FIGURE_INDEX[
        [
            "Figure ID",
            "Filename",
            "Width",
            "Height",
            "Verification"
        ]
    ].to_string(index=False)
)

print("\nFIGURE INDEX EXPORT")
print("NSL12_final_figure_index.csv:", export_status)

# ----------------------------------------------------------------------
# 4. Final figure checks
# ----------------------------------------------------------------------

assert len(NSL12_FIGURE_INDEX) == 5
assert NSL12_FIGURE_INDEX["Verification"].eq("PASS").all()

print("\nFIGURE INTEGRITY")
print("Verified figures:", len(NSL12_FIGURE_INDEX))
print("PNG dimensions: PASS")
print("SHA-256 fingerprints: RECORDED")
print("Figure index: PASS")
print("Original figures modified: NO")
print("New figure generation: NO")

print("\nCELL 05: PASSED")
print("Dissertation figure index: VERIFIED")
print("Model training performed: NO")
print("=" * 72)

CELL 05 — FINAL FIGURE AND VISUAL-EVIDENCE INDEX

VERIFIED RESEARCH FIGURES
Figure ID                                  Filename  Width  Height Verification
   FIG-01 NSL11_within_benchmark_metric_effects.png   1980    1196         PASS
   FIG-02 NSL11_nsl_validation_vs_test_class_f1.png   2179    1294         PASS
   FIG-03  NSL11_nsl_attack_family_distribution.png   1980    1179         PASS
   FIG-04     NSL11_nsl_attack_novelty_accuracy.png   1780    1196         PASS
   FIG-05       NSL11_nsl_family_novelty_recall.png   2180    1196         PASS

FIGURE INDEX EXPORT
NSL12_final_figure_index.csv: CREATED

FIGURE INTEGRITY
Verified figures: 5
PNG dimensions: PASS
SHA-256 fingerprints: RECORDED
Figure index: PASS
Original figures modified: NO
New figure generation: NO

CELL 05: PASSED
Dissertation figure index: VERIFIED
Model training performed: NO


In [6]:
# ======================================================================
# CELL 06 — RESEARCH QUESTION AND OBJECTIVES EVIDENCE MAPPING
# ======================================================================

import json
import pandas as pd

print("=" * 72)
print("CELL 06 — RESEARCH QUESTION AND OBJECTIVES EVIDENCE MAPPING")
print("=" * 72)

assert "NSL12_FINDINGS_REGISTER" in globals()
assert "NSL12_OUTPUT_DIR" in globals()
assert "ns12_export_table" in globals()

# ----------------------------------------------------------------------
# 1. Approved research question
# ----------------------------------------------------------------------

NSL12_RESEARCH_QUESTION = (
    "How can Random Forest be enhanced to improve robustness "
    "and generalisation for network intrusion detection under "
    "class imbalance and cross-dataset conditions?"
)

# ----------------------------------------------------------------------
# 2. Approved research objectives
# ----------------------------------------------------------------------

NSL12_OBJECTIVES = {
    "O1": (
        "To critically examine the limitations of standard "
        "Random Forest for network intrusion detection."
    ),
    "O2": (
        "To design and develop an enhanced Random Forest-based "
        "intrusion detection framework."
    ),
    "O3": (
        "To evaluate whether the enhanced Random Forest framework "
        "improves detection performance and robustness compared "
        "with a standard Random Forest baseline."
    )
}

# ----------------------------------------------------------------------
# 3. Research evidence mapping
# ----------------------------------------------------------------------

mapping_records = [
    {
        "Research Element": "RQ",
        "Research Statement": NSL12_RESEARCH_QUESTION,
        "Finding IDs": "F01, F02, F03, F04, F05, F06, F07, F08, F09, F10",
        "Evidence Summary": (
            "CIC enhancement improved Macro F1 by 0.018702 "
            "and Macro Recall by 0.048248. "
            "The NSL frozen depth configuration produced no "
            "prediction changes. NSL official-test performance "
            "declined substantially relative to validation."
        ),
        "Assessment": "Answered with qualifications",
        "Interpretation Boundary": (
            "Benchmark-specific improvement does not establish "
            "successful direct cross-dataset generalisation."
        )
    },
    {
        "Research Element": "O1",
        "Research Statement": NSL12_OBJECTIVES["O1"],
        "Finding IDs": "F03, F06, F07, F09, F10",
        "Evidence Summary": (
            "The standard Random Forest showed limitations "
            "in minority-class detection and evaluation under "
            "distribution shift and unseen attack types."
        ),
        "Assessment": "Addressed",
        "Interpretation Boundary": (
            "Rare-class sample sizes and dataset characteristics "
            "limit the generality of the observations."
        )
    },
    {
        "Research Element": "O2",
        "Research Statement": NSL12_OBJECTIVES["O2"],
        "Finding IDs": "F01, F02, F03, F05",
        "Evidence Summary": (
            "An enhanced Random Forest configuration was "
            "developed and evaluated on CIC-IDS2017. "
            "A frozen depth configuration was independently "
            "replicated on NSL-KDD."
        ),
        "Assessment": "Implemented",
        "Interpretation Boundary": (
            "The NSL-KDD experiment is an independent "
            "methodological replication, not a transfer "
            "of the fitted CIC model."
        )
    },
    {
        "Research Element": "O3",
        "Research Statement": NSL12_OBJECTIVES["O3"],
        "Finding IDs": "F01, F02, F03, F04, F05, F06, F07, F08, F09, F10",
        "Evidence Summary": (
            "The CIC enhanced configuration improved Macro F1 "
            "and Macro Recall, with mixed class-level effects. "
            "McNemar testing did not identify a statistically "
            "significant difference in overall correctness. "
            "NSL models produced identical predictions, while "
            "official-test performance revealed generalisation "
            "limitations."
        ),
        "Assessment": "Evaluated with qualifications",
        "Interpretation Boundary": (
            "The evidence supports a CIC-specific improvement, "
            "but not a general improvement across both benchmarks."
        )
    }
]

NSL12_RESEARCH_MAPPING = pd.DataFrame(mapping_records)

# ----------------------------------------------------------------------
# 4. Validate references to research findings
# ----------------------------------------------------------------------

valid_findings = set(
    NSL12_FINDINGS_REGISTER["Finding ID"]
)

assert valid_findings == {
    f"F{i:02d}" for i in range(1, 11)
}

for row in mapping_records:

    referenced = {
        item.strip()
        for item in row["Finding IDs"].split(",")
    }

    assert referenced.issubset(valid_findings), (
        "Unrecognised finding reference in "
        + row["Research Element"]
    )

assert len(NSL12_RESEARCH_MAPPING) == 4

# ----------------------------------------------------------------------
# 5. Export research mapping
# ----------------------------------------------------------------------

mapping_path, mapping_status = ns12_export_table(
    NSL12_RESEARCH_MAPPING,
    "NSL12_research_question_objectives_mapping.csv"
)

# ----------------------------------------------------------------------
# 6. Prepare carefully bounded research conclusion
# ----------------------------------------------------------------------

NSL12_RESEARCH_CONCLUSION = {
    "research_question": NSL12_RESEARCH_QUESTION,

    "main_conclusion": (
        "The enhanced Random Forest configuration demonstrated "
        "a measurable improvement in class-balanced detection "
        "performance within CIC-IDS2017, particularly in Macro "
        "Recall and Macro F1. However, the evidence does not "
        "establish universal robustness or successful "
        "cross-dataset generalisation."
    ),

    "cic_macro_f1_change": 0.018702,
    "cic_macro_recall_change": 0.048248,
    "cic_mcnemar_p_value": 0.5384046,

    "nsl_macro_f1_change": 0.0,
    "nsl_validation_macro_f1": 0.958147,
    "nsl_official_test_macro_f1": 0.504649,

    "nsl_test_only_attack_records": 3750,
    "nsl_test_only_attack_errors": 3051,

    "methodological_scope": (
        "Independent benchmark-specific Random Forest "
        "experiments; no direct transfer of fitted models."
    ),

    "statistical_scope": (
        "McNemar testing concerns paired overall correctness, "
        "not the statistical significance of Macro F1 changes."
    ),

    "generalisation_scope": (
        "Observed NSL validation-to-test deterioration cannot "
        "be attributed to a single cause."
    )
}

conclusion_path = (
    NSL12_OUTPUT_DIR
    / "NSL12_research_conclusion.json"
)

if conclusion_path.exists():

    with open(conclusion_path, "r", encoding="utf-8") as f:
        existing_conclusion = json.load(f)

    assert existing_conclusion == NSL12_RESEARCH_CONCLUSION
    conclusion_status = "VERIFIED EXISTING"

else:

    with open(conclusion_path, "w", encoding="utf-8") as f:
        json.dump(
            NSL12_RESEARCH_CONCLUSION,
            f,
            indent=2,
            ensure_ascii=False
        )

    conclusion_status = "CREATED"

NSL12_CELL06_FILES = [
    mapping_path,
    conclusion_path
]

# ----------------------------------------------------------------------
# 7. Final verification
# ----------------------------------------------------------------------

print("\nRESEARCH OBJECTIVES")

for objective_id, statement in NSL12_OBJECTIVES.items():
    print(f"{objective_id}: {statement}")

print("\nRESEARCH EVIDENCE MAPPING")

print(
    NSL12_RESEARCH_MAPPING[
        [
            "Research Element",
            "Finding IDs",
            "Assessment"
        ]
    ].to_string(index=False)
)

print("\nRESEARCH EXPORTS")
print(
    "NSL12_research_question_objectives_mapping.csv:",
    mapping_status
)
print(
    "NSL12_research_conclusion.json:",
    conclusion_status
)

print("\nRESEARCH INTEGRITY")
print("Research question: VERIFIED")
print("Research objectives:", len(NSL12_OBJECTIVES))
print("Evidence mappings:", len(NSL12_RESEARCH_MAPPING))
print("Research finding references: PASS")
print("Methodological boundaries: PRESERVED")
print("Cross-dataset transfer claim: EXCLUDED")
print("Source experiments modified: NO")

print("\nCELL 06: PASSED")
print("Research evidence mapping: COMPLETE")
print("Model training performed: NO")
print("=" * 72)

CELL 06 — RESEARCH QUESTION AND OBJECTIVES EVIDENCE MAPPING

RESEARCH OBJECTIVES
O1: To critically examine the limitations of standard Random Forest for network intrusion detection.
O2: To design and develop an enhanced Random Forest-based intrusion detection framework.
O3: To evaluate whether the enhanced Random Forest framework improves detection performance and robustness compared with a standard Random Forest baseline.

RESEARCH EVIDENCE MAPPING
Research Element                                      Finding IDs                    Assessment
              RQ F01, F02, F03, F04, F05, F06, F07, F08, F09, F10  Answered with qualifications
              O1                          F03, F06, F07, F09, F10                     Addressed
              O2                               F01, F02, F03, F05                   Implemented
              O3 F01, F02, F03, F04, F05, F06, F07, F08, F09, F10 Evaluated with qualifications

RESEARCH EXPORTS
NSL12_research_question_objectives_mapping.csv: 

In [7]:
# ======================================================================
# CELL 07 — DISSERTATION EVIDENCE CATALOGUE AND TRACEABILITY
# ======================================================================

from pathlib import Path
import hashlib
import json
import pandas as pd

print("=" * 72)
print("CELL 07 — DISSERTATION EVIDENCE CATALOGUE")
print("=" * 72)

# ----------------------------------------------------------------------
# 1. Verify prerequisites
# ----------------------------------------------------------------------

required = [
    "PROJECT_ROOT",
    "NSL12_OUTPUT_DIR",
    "NSL12_CELL04_FILES",
    "NSL12_CELL05_FILES",
    "NSL12_CELL06_FILES",
    "ns12_export_table"
]

missing = [name for name in required if name not in globals()]

if missing:
    raise RuntimeError(
        "Missing prerequisites: " + ", ".join(missing)
    )

NSL12_OUTPUT_DIR = Path(NSL12_OUTPUT_DIR).resolve()

evidence_paths = [
    Path(p).resolve()
    for p in (
        NSL12_CELL04_FILES
        + NSL12_CELL05_FILES
        + NSL12_CELL06_FILES
    )
]

assert len(evidence_paths) == 12
assert len(set(evidence_paths)) == 12

# ----------------------------------------------------------------------
# 2. Define evidence roles
# ----------------------------------------------------------------------

evidence_roles = {
    "NSL12_final_model_results.csv":
        ("Model performance comparison", "Results"),

    "NSL12_within_benchmark_effects.csv":
        ("Benchmark-specific enhancement effects", "Results"),

    "NSL12_class_level_effects.csv":
        ("Class-level detection trade-offs", "Results"),

    "NSL12_nsl_class_generalisation.csv":
        ("NSL validation-to-test class performance", "Results"),

    "NSL12_nsl_family_distribution_shift.csv":
        ("Attack-family distribution shift", "Results"),

    "NSL12_nsl_attack_novelty.csv":
        ("Attack-novelty cohort performance", "Results"),

    "NSL12_nsl_test_only_attacks.csv":
        ("Test-only attack inventory", "Results"),

    "NSL12_research_findings.csv":
        ("Consolidated research findings", "Discussion"),

    "NSL12_interpretation_boundaries.csv":
        ("Research limitations and interpretation boundaries", "Discussion"),

    "NSL12_final_figure_index.csv":
        ("Research figure references", "Results"),

    "NSL12_research_question_objectives_mapping.csv":
        ("Research question and objective traceability", "Discussion"),

    "NSL12_research_conclusion.json":
        ("Evidence-based research conclusion", "Discussion")
}

assert {p.name for p in evidence_paths} == set(evidence_roles)

# ----------------------------------------------------------------------
# 3. File fingerprint utility
# ----------------------------------------------------------------------

def ns12_file_sha256(path):

    digest = hashlib.sha256()

    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)

    return digest.hexdigest()

# ----------------------------------------------------------------------
# 4. Construct evidence catalogue
# ----------------------------------------------------------------------

catalogue_records = []

for index, path in enumerate(
    sorted(evidence_paths, key=lambda p: p.name),
    start=1
):

    assert path.is_file()
    assert path.parent == NSL12_OUTPUT_DIR
    assert path.stat().st_size > 0

    role, section = evidence_roles[path.name]

    if path.suffix.lower() == ".csv":

        content = pd.read_csv(path)
        record_count = len(content)
        file_type = "CSV"

    elif path.suffix.lower() == ".json":

        with open(path, "r", encoding="utf-8") as f:
            content = json.load(f)

        assert isinstance(content, dict)
        record_count = len(content)
        file_type = "JSON"

    else:
        raise ValueError(
            f"Unexpected evidence file type: {path.name}"
        )

    catalogue_records.append({
        "Evidence ID": f"E{index:02d}",
        "Filename": path.name,
        "File Type": file_type,
        "Research Role": role,
        "Suggested Section": section,
        "Records": record_count,
        "Size Bytes": path.stat().st_size,
        "SHA256": ns12_file_sha256(path),
        "Verification": "PASS"
    })

NSL12_EVIDENCE_CATALOGUE = pd.DataFrame(catalogue_records)

# ----------------------------------------------------------------------
# 5. Export catalogue
# ----------------------------------------------------------------------

catalogue_path, catalogue_status = ns12_export_table(
    NSL12_EVIDENCE_CATALOGUE,
    "NSL12_dissertation_evidence_catalogue.csv"
)

NSL12_CELL07_FILES = [catalogue_path]

# ----------------------------------------------------------------------
# 6. Verification
# ----------------------------------------------------------------------

assert len(NSL12_EVIDENCE_CATALOGUE) == 12

assert (
    NSL12_EVIDENCE_CATALOGUE["Verification"] == "PASS"
).all()

print("\nDISSERTATION EVIDENCE CATALOGUE")

print(
    NSL12_EVIDENCE_CATALOGUE[
        [
            "Evidence ID",
            "Filename",
            "File Type",
            "Records",
            "Verification"
        ]
    ].to_string(index=False)
)

print("\nCATALOGUE EXPORT")
print(
    "NSL12_dissertation_evidence_catalogue.csv:",
    catalogue_status
)

print("\nEVIDENCE INTEGRITY")
print("Evidence files catalogued:", len(evidence_paths))
print("CSV files:", sum(p.suffix == ".csv" for p in evidence_paths))
print("JSON files:", sum(p.suffix == ".json" for p in evidence_paths))
print("File readability: PASS")
print("SHA-256 fingerprints: RECORDED")
print("Evidence traceability: PASS")
print("Original experiments modified: NO")

print("\nCELL 07: PASSED")
print("Dissertation evidence catalogue: COMPLETE")
print("Model training performed: NO")
print("=" * 72)

CELL 07 — DISSERTATION EVIDENCE CATALOGUE

DISSERTATION EVIDENCE CATALOGUE
Evidence ID                                       Filename File Type  Records Verification
        E01                  NSL12_class_level_effects.csv       CSV       20         PASS
        E02                   NSL12_final_figure_index.csv       CSV        5         PASS
        E03                  NSL12_final_model_results.csv       CSV        4         PASS
        E04            NSL12_interpretation_boundaries.csv       CSV        9         PASS
        E05                   NSL12_nsl_attack_novelty.csv       CSV        2         PASS
        E06             NSL12_nsl_class_generalisation.csv       CSV       10         PASS
        E07        NSL12_nsl_family_distribution_shift.csv       CSV        5         PASS
        E08                NSL12_nsl_test_only_attacks.csv       CSV       17         PASS
        E09                 NSL12_research_conclusion.json      JSON       13         PASS
        E10    

In [8]:
# ======================================================================
# CELL 08 — DISSERTATION INTERPRETATION AND CHAPTER EVIDENCE MAP
# ======================================================================

from pathlib import Path
import pandas as pd

print("=" * 72)
print("CELL 08 — DISSERTATION INTERPRETATION PACKAGE")
print("=" * 72)

# ----------------------------------------------------------------------
# 1. Verify prerequisites
# ----------------------------------------------------------------------

required = [
    "NSL12_MODEL_REGISTER",
    "NSL12_FINAL_EFFECTS",
    "NSL12_RESEARCH_CONCLUSION",
    "NSL12_EVIDENCE_CATALOGUE",
    "NSL12_OUTPUT_DIR",
    "ns12_export_table"
]

missing = [name for name in required if name not in globals()]

if missing:
    raise RuntimeError(
        "Missing prerequisites: " + ", ".join(missing)
    )

models = NSL12_MODEL_REGISTER.set_index("Model")

required_models = [
    "RF-B-CIC",
    "RF-E-CIC",
    "RF-B-NSL",
    "RF-H2-NSL"
]

assert set(required_models).issubset(models.index)

# ----------------------------------------------------------------------
# 2. Extract verified model metrics
# ----------------------------------------------------------------------

cic_baseline_f1 = float(models.loc["RF-B-CIC", "Macro F1"])
cic_enhanced_f1 = float(models.loc["RF-E-CIC", "Macro F1"])

cic_baseline_recall = float(
    models.loc["RF-B-CIC", "Macro Recall"]
)

cic_enhanced_recall = float(
    models.loc["RF-E-CIC", "Macro Recall"]
)

cic_baseline_precision = float(
    models.loc["RF-B-CIC", "Macro Precision"]
)

cic_enhanced_precision = float(
    models.loc["RF-E-CIC", "Macro Precision"]
)

nsl_baseline_f1 = float(
    models.loc["RF-B-NSL", "Macro F1"]
)

nsl_comparison_f1 = float(
    models.loc["RF-H2-NSL", "Macro F1"]
)

cic_f1_change = cic_enhanced_f1 - cic_baseline_f1
cic_recall_change = (
    cic_enhanced_recall - cic_baseline_recall
)

cic_precision_change = (
    cic_enhanced_precision - cic_baseline_precision
)

nsl_f1_change = nsl_comparison_f1 - nsl_baseline_f1

# Values are displayed to six decimal places in prior records.
assert abs(cic_f1_change - 0.018702) < 0.000005
assert abs(cic_recall_change - 0.048248) < 0.000005
assert abs(nsl_f1_change) < 1e-12

# ----------------------------------------------------------------------
# 3. Build dissertation interpretation summary
# ----------------------------------------------------------------------

NSL12_DISSERTATION_SUMMARY = f"""# Final Research Evidence Summary

## 1. CIC-IDS2017 Performance

The standard Random Forest baseline achieved a test Macro F1
of {cic_baseline_f1:.6f}. The enhanced Random Forest achieved
{cic_enhanced_f1:.6f}, representing a change of
{cic_f1_change:+.6f}.

Macro Recall changed from {cic_baseline_recall:.6f} to
{cic_enhanced_recall:.6f}, a difference of
{cic_recall_change:+.6f}.

Macro Precision changed from {cic_baseline_precision:.6f}
to {cic_enhanced_precision:.6f}, a difference of
{cic_precision_change:+.6f}.

These findings show an improvement in class-balanced
detection performance, accompanied by precision and
class-specific trade-offs.

The CIC comparison evaluates the combined enhancement
configuration. Its observed effect cannot be attributed
exclusively to one enhancement component.

## 2. Statistical Evaluation

The archived CIC paired-correctness comparison produced
a McNemar p-value of 0.5384046.

At the conventional 0.05 significance threshold,
this does not provide evidence of a statistically
significant difference in overall classification
correctness.

This result does not establish whether the observed
Macro F1 difference is statistically significant.

## 3. NSL-KDD Replication

The NSL-KDD baseline achieved official-test Macro F1
of {nsl_baseline_f1:.6f}.

The frozen maximum-depth configuration achieved
{nsl_comparison_f1:.6f}.

The difference was {nsl_f1_change:+.6f}.

Both models generated identical validation and
official-test class predictions.

This establishes no observed classification benefit
from the frozen depth restriction in this experiment.

## 4. Generalisation Findings

NSL-KDD validation Macro F1 was 0.958147,
compared with an official-test Macro F1 of 0.504649.

The difference was approximately -0.453498.

Official-test R2L recall was 0.054766,
while U2R recall was 0.029851.

These results demonstrate substantial performance
deterioration on the official test partition,
particularly for minority attack families.

## 5. Attack Novelty and Distribution Shift

The NSL-KDD official test partition contained
3,750 observations from 17 attack names absent
from the development training data.

The test-only attack-name cohort achieved an
accuracy of 0.186400.

These observations contributed 3,051 of the
5,447 official-test classification errors,
representing approximately 56.012%.

Attack novelty and changes in class composition
are associated with the observed deterioration.

However, the novelty cohorts have different
attack-family compositions, so the performance
difference cannot be attributed solely to novelty.

## 6. Overall Research Interpretation

The evidence supports a benchmark-specific
improvement in class-balanced detection
performance for the enhanced CIC-IDS2017
Random Forest configuration.

The independent NSL-KDD experiment did not
demonstrate an improvement from its frozen
maximum-depth restriction.

NSL-KDD results further demonstrate substantial
limitations when evaluated on its official
test distribution.

Therefore, the findings support conditional
improvements within the evaluated CIC benchmark,
but do not establish universal robustness
or successful direct cross-dataset generalisation.

## 7. Research Boundaries

- CIC-IDS2017 and NSL-KDD have different target label spaces.
- Their raw Macro F1 scores must not be ranked as equivalent tasks.
- No fitted CIC model was transferred directly to NSL-KDD.
- The NSL experiment evaluates independent methodological replication.
- Rare-class findings require caution because of small sample sizes.
- The NSL generalisation gap cannot be assigned to a single factor.
- Test-set reuse and possible network-flow dependence limit
  confirmatory interpretation.

This summary is an evidence-backed dissertation working draft.
Its final wording should be integrated with the dissertation's
methodology, literature and approved chapter structure.
"""

# ----------------------------------------------------------------------
# 4. Chapter evidence index
# ----------------------------------------------------------------------

section_records = [
    {
        "Suggested Section": "Results — Model Performance",
        "Research Focus": "Baseline and comparison models",
        "Evidence Files": (
            "NSL12_final_model_results.csv; "
            "NSL12_within_benchmark_effects.csv"
        )
    },
    {
        "Suggested Section": "Results — Class-Level Detection",
        "Research Focus": "Class-level performance trade-offs",
        "Evidence Files": (
            "NSL12_class_level_effects.csv; "
            "NSL12_nsl_class_generalisation.csv"
        )
    },
    {
        "Suggested Section": "Results — Distribution Shift",
        "Research Focus": "Generalisation and attack-family prevalence",
        "Evidence Files": (
            "NSL12_nsl_family_distribution_shift.csv; "
            "NSL12_nsl_class_generalisation.csv"
        )
    },
    {
        "Suggested Section": "Results — Attack Novelty",
        "Research Focus": "Training-seen and test-only attack cohorts",
        "Evidence Files": (
            "NSL12_nsl_attack_novelty.csv; "
            "NSL12_nsl_test_only_attacks.csv"
        )
    },
    {
        "Suggested Section": "Discussion — Research Findings",
        "Research Focus": "Research question and objectives",
        "Evidence Files": (
            "NSL12_research_findings.csv; "
            "NSL12_research_question_objectives_mapping.csv"
        )
    },
    {
        "Suggested Section": "Discussion — Limitations",
        "Research Focus": "Evidence boundaries and conclusions",
        "Evidence Files": (
            "NSL12_interpretation_boundaries.csv; "
            "NSL12_research_conclusion.json"
        )
    }
]

NSL12_SECTION_INDEX = pd.DataFrame(section_records)

known_files = set(
    NSL12_EVIDENCE_CATALOGUE["Filename"]
)

for record in section_records:

    files = [
        name.strip()
        for name in record["Evidence Files"].split(";")
    ]

    assert set(files).issubset(known_files)

# ----------------------------------------------------------------------
# 5. Export chapter evidence index
# ----------------------------------------------------------------------

section_path, section_status = ns12_export_table(
    NSL12_SECTION_INDEX,
    "NSL12_dissertation_section_evidence_map.csv"
)

# ----------------------------------------------------------------------
# 6. Export interpretation summary safely
# ----------------------------------------------------------------------

summary_path = (
    Path(NSL12_OUTPUT_DIR)
    / "NSL12_dissertation_evidence_summary.md"
)

if summary_path.exists():

    existing = summary_path.read_text(encoding="utf-8")

    assert existing == NSL12_DISSERTATION_SUMMARY

    summary_status = "VERIFIED EXISTING"

else:

    summary_path.write_text(
        NSL12_DISSERTATION_SUMMARY,
        encoding="utf-8"
    )

    summary_status = "CREATED"

NSL12_CELL08_FILES = [
    section_path,
    summary_path
]

# ----------------------------------------------------------------------
# 7. Verification
# ----------------------------------------------------------------------

assert len(NSL12_SECTION_INDEX) == 6

print("\nCHAPTER EVIDENCE INDEX")

print(
    NSL12_SECTION_INDEX[
        ["Suggested Section", "Research Focus"]
    ].to_string(index=False)
)

print("\nRESEARCH SUMMARY")
print("CIC Macro F1 change:", f"{cic_f1_change:+.6f}")
print("CIC Macro Recall change:", f"{cic_recall_change:+.6f}")
print("CIC Macro Precision change:", f"{cic_precision_change:+.6f}")
print("NSL Macro F1 change:", f"{nsl_f1_change:+.6f}")

print("\nEXPORTS")
print(
    "NSL12_dissertation_section_evidence_map.csv:",
    section_status
)
print(
    "NSL12_dissertation_evidence_summary.md:",
    summary_status
)

print("\nINTERPRETATION INTEGRITY")
print("Chapter evidence sections:", len(NSL12_SECTION_INDEX))
print("Research evidence references: PASS")
print("Benchmark distinction: PRESERVED")
print("Statistical interpretation: PRESERVED")
print("Generalisation boundaries: PRESERVED")
print("Existing experiments modified: NO")

print("\nCELL 08: PASSED")
print("Dissertation interpretation package: COMPLETE")
print("Model training performed: NO")
print("=" * 72)

CELL 08 — DISSERTATION INTERPRETATION PACKAGE

CHAPTER EVIDENCE INDEX
              Suggested Section                              Research Focus
    Results — Model Performance              Baseline and comparison models
Results — Class-Level Detection          Class-level performance trade-offs
   Results — Distribution Shift Generalisation and attack-family prevalence
       Results — Attack Novelty  Training-seen and test-only attack cohorts
 Discussion — Research Findings            Research question and objectives
       Discussion — Limitations         Evidence boundaries and conclusions

RESEARCH SUMMARY
CIC Macro F1 change: +0.018702
CIC Macro Recall change: +0.048248
CIC Macro Precision change: -0.034275
NSL Macro F1 change: +0.000000

EXPORTS
NSL12_dissertation_section_evidence_map.csv: CREATED
NSL12_dissertation_evidence_summary.md: CREATED

INTERPRETATION INTEGRITY
Chapter evidence sections: 6
Research evidence references: PASS
Benchmark distinction: PRESERVED
Statistical 

In [9]:
# ======================================================================
# CELL 09 — INDEPENDENT EXPORT VERIFICATION AND INTEGRITY AUDIT
# ======================================================================

from pathlib import Path
import json
import pandas as pd

print("=" * 72)
print("CELL 09 — INDEPENDENT EXPORT VERIFICATION")
print("=" * 72)

# ----------------------------------------------------------------------
# 1. Verify required outputs
# ----------------------------------------------------------------------

required = [
    "NSL12_CELL04_FILES",
    "NSL12_CELL05_FILES",
    "NSL12_CELL06_FILES",
    "NSL12_CELL07_FILES",
    "NSL12_CELL08_FILES",
    "NSL12_EVIDENCE_CATALOGUE",
    "NSL12_OUTPUT_DIR",
    "NSL12_SOURCE_DIR",
    "ns12_file_sha256"
]

missing = [name for name in required if name not in globals()]

if missing:
    raise RuntimeError(
        "Missing prerequisites: " + ", ".join(missing)
    )

all_evidence_files = [
    Path(p).resolve()
    for p in (
        NSL12_CELL04_FILES
        + NSL12_CELL05_FILES
        + NSL12_CELL06_FILES
        + NSL12_CELL07_FILES
        + NSL12_CELL08_FILES
    )
]

assert len(all_evidence_files) == 15
assert len(set(all_evidence_files)) == 15

for path in all_evidence_files:

    assert path.is_file()
    assert path.parent == Path(NSL12_OUTPUT_DIR).resolve()
    assert path.stat().st_size > 0

# ----------------------------------------------------------------------
# 2. Independently reload the evidence catalogue
# ----------------------------------------------------------------------

catalogue = pd.read_csv(
    Path(NSL12_OUTPUT_DIR)
    / "NSL12_dissertation_evidence_catalogue.csv"
)

assert len(catalogue) == 12

for _, row in catalogue.iterrows():

    path = Path(NSL12_OUTPUT_DIR) / row["Filename"]

    assert path.is_file()

    actual_hash = ns12_file_sha256(path)

    assert actual_hash == row["SHA256"], (
        "Evidence fingerprint mismatch: " + path.name
    )

    if path.suffix.lower() == ".csv":

        loaded = pd.read_csv(path)

        assert len(loaded) == int(row["Records"])

    elif path.suffix.lower() == ".json":

        with open(path, "r", encoding="utf-8") as f:
            loaded = json.load(f)

        assert len(loaded) == int(row["Records"])

print("\nEVIDENCE CATALOGUE VERIFICATION")
print("Catalogued files:", len(catalogue))
print("Saved-file fingerprints: PASS")
print("Saved-file content counts: PASS")

# ----------------------------------------------------------------------
# 3. Reconcile final model results
# ----------------------------------------------------------------------

model_table = pd.read_csv(
    Path(NSL12_OUTPUT_DIR)
    / "NSL12_final_model_results.csv"
)

effect_table = pd.read_csv(
    Path(NSL12_OUTPUT_DIR)
    / "NSL12_within_benchmark_effects.csv"
)

assert len(model_table) == 4
assert len(effect_table) == 14

models = model_table.set_index("Model")

# Check identical NSL aggregate results.
for metric in [
    "Accuracy",
    "Macro Precision",
    "Macro Recall",
    "Macro F1",
    "MCC"
]:

    baseline = float(
        models.loc["RF-B-NSL", metric]
    )

    comparison = float(
        models.loc["RF-H2-NSL", metric]
    )

    assert abs(baseline - comparison) < 1e-12

# Reconcile CIC Macro F1 effect.
cic_calculated_change = (
    float(models.loc["RF-E-CIC", "Macro F1"])
    - float(models.loc["RF-B-CIC", "Macro F1"])
)

matching_effect = effect_table.loc[
    (effect_table["Benchmark"] == "CIC-IDS2017")
    & (effect_table["Metric"] == "Macro F1"),
    "Change"
]

assert len(matching_effect) == 1

cic_recorded_change = float(
    matching_effect.iloc[0]
)

assert abs(
    cic_calculated_change - cic_recorded_change
) < 0.000005

print("\nNUMERICAL RECONCILIATION")
print("Model records: 4")
print("Within-benchmark effects: 14")
print("CIC Macro F1 effect: PASS")
print("NSL aggregate model agreement: PASS")

# ----------------------------------------------------------------------
# 4. Verify archived research figures against recorded hashes
# ----------------------------------------------------------------------

figure_index = pd.read_csv(
    Path(NSL12_OUTPUT_DIR)
    / "NSL12_final_figure_index.csv"
)

assert len(figure_index) == 5

for _, row in figure_index.iterrows():

    figure_path = (
        Path(PROJECT_ROOT) / row["Relative Path"]
    ).resolve()

    assert figure_path.is_file()

    actual_hash = ns12_file_sha256(figure_path)

    assert actual_hash == row["SHA256"], (
        "Figure fingerprint mismatch: " + figure_path.name
    )

print("\nFIGURE VERIFICATION")
print("Archived figures:", len(figure_index))
print("Figure fingerprint comparison: PASS")
print("Original figures modified by audit: NO")

# ----------------------------------------------------------------------
# 5. Verify research findings and objective references
# ----------------------------------------------------------------------

findings = pd.read_csv(
    Path(NSL12_OUTPUT_DIR)
    / "NSL12_research_findings.csv"
)

mapping = pd.read_csv(
    Path(NSL12_OUTPUT_DIR)
    / "NSL12_research_question_objectives_mapping.csv"
)

assert len(findings) == 10
assert len(mapping) == 4

valid_finding_ids = set(findings["Finding ID"])

for _, row in mapping.iterrows():

    referenced_ids = {
        value.strip()
        for value in row["Finding IDs"].split(",")
    }

    assert referenced_ids.issubset(valid_finding_ids)

print("\nRESEARCH TRACEABILITY")
print("Research findings:", len(findings))
print("Research mappings:", len(mapping))
print("Evidence references: PASS")

# ----------------------------------------------------------------------
# 6. Construct an export-audit record
# ----------------------------------------------------------------------

audited_fingerprints = {
    path.name: ns12_file_sha256(path)
    for path in sorted(
        all_evidence_files,
        key=lambda p: p.name
    )
}

NSL12_EXPORT_AUDIT = {
    "notebook": "12_Final_Research_Evidence_Consolidation",
    "audit_stage": "CELL_09",
    "status": "PASS",
    "audited_evidence_files": len(all_evidence_files),
    "catalogued_files": len(catalogue),
    "verified_figures": len(figure_index),
    "research_findings": len(findings),
    "research_mappings": len(mapping),
    "model_result_reconciliation": "PASS",
    "benchmark_effect_reconciliation": "PASS",
    "figure_fingerprint_verification": "PASS",
    "research_traceability": "PASS",
    "additional_model_training": False,
    "source_experiment_modification": False,
    "file_fingerprints": audited_fingerprints
}

audit_path = (
    Path(NSL12_OUTPUT_DIR)
    / "NSL12_export_integrity_audit.json"
)

if audit_path.exists():

    with open(audit_path, "r", encoding="utf-8") as f:
        previous_audit = json.load(f)

    assert previous_audit == NSL12_EXPORT_AUDIT

    audit_status = "VERIFIED EXISTING"

else:

    with open(audit_path, "w", encoding="utf-8") as f:
        json.dump(
            NSL12_EXPORT_AUDIT,
            f,
            indent=2,
            ensure_ascii=False
        )

    audit_status = "CREATED"

NSL12_CELL09_FILES = [audit_path]

# ----------------------------------------------------------------------
# 7. Final export verification
# ----------------------------------------------------------------------

assert NSL12_EXPORT_AUDIT["status"] == "PASS"

print("\nEXPORT AUDIT")
print("Audited evidence files:", len(all_evidence_files))
print("Evidence SHA-256 verification: PASS")
print("Model-result reconciliation: PASS")
print("Figure reference verification: PASS")
print("Research evidence traceability: PASS")
print("Audit record:", audit_status)
print("Source experiments modified: NO")

print("\nCELL 09: PASSED")
print("Independent export verification: COMPLETE")
print("Model training performed: NO")
print("=" * 72)

CELL 09 — INDEPENDENT EXPORT VERIFICATION

EVIDENCE CATALOGUE VERIFICATION
Catalogued files: 12
Saved-file fingerprints: PASS
Saved-file content counts: PASS

NUMERICAL RECONCILIATION
Model records: 4
Within-benchmark effects: 14
CIC Macro F1 effect: PASS
NSL aggregate model agreement: PASS

FIGURE VERIFICATION
Archived figures: 5
Figure fingerprint comparison: PASS
Original figures modified by audit: NO

RESEARCH TRACEABILITY
Research findings: 10
Research mappings: 4
Evidence references: PASS

EXPORT AUDIT
Audited evidence files: 15
Evidence SHA-256 verification: PASS
Model-result reconciliation: PASS
Figure reference verification: PASS
Research evidence traceability: PASS
Audit record: CREATED
Source experiments modified: NO

CELL 09: PASSED
Independent export verification: COMPLETE
Model training performed: NO


In [10]:
# ======================================================================
# CELL 10 — FINAL RESEARCH-EVIDENCE INTEGRITY
# ======================================================================

from pathlib import Path
import json
import pandas as pd

print("=" * 72)
print("CELL 10 — FINAL RESEARCH-EVIDENCE INTEGRITY")
print("=" * 72)

required = [
    "PROJECT_ROOT",
    "NSL12_OUTPUT_DIR",
    "NSL12_EXPORT_AUDIT",
    "NSL12_CELL09_FILES",
    "NSL12_RESEARCH_CONCLUSION",
    "ns12_file_sha256"
]

missing = [name for name in required if name not in globals()]

if missing:
    raise RuntimeError(
        "Missing prerequisites: " + ", ".join(missing)
    )

output_dir = Path(NSL12_OUTPUT_DIR).resolve()

# ----------------------------------------------------------------------
# 1. Verify previous export-audit fingerprints
# ----------------------------------------------------------------------

previous_fingerprints = NSL12_EXPORT_AUDIT["file_fingerprints"]

assert len(previous_fingerprints) == 15

for filename, expected_hash in previous_fingerprints.items():

    path = (output_dir / filename).resolve()

    assert path.parent == output_dir
    assert path.is_file()
    assert ns12_file_sha256(path) == expected_hash

audit_path = Path(NSL12_CELL09_FILES[0]).resolve()

assert audit_path.is_file()

print("\nPRIOR EVIDENCE AUDIT")
print("Evidence files reverified:", len(previous_fingerprints))
print("Export audit record: PASS")
print("SHA-256 verification: PASS")

# ----------------------------------------------------------------------
# 2. Verify final research figures
# ----------------------------------------------------------------------

figure_index = pd.read_csv(
    output_dir / "NSL12_final_figure_index.csv"
)

assert len(figure_index) == 5

for _, row in figure_index.iterrows():

    figure_path = (
        Path(PROJECT_ROOT) / row["Relative Path"]
    ).resolve()

    assert figure_path.is_file()

    assert (
        ns12_file_sha256(figure_path)
        == row["SHA256"]
    )

print("\nRESEARCH FIGURES")
print("Verified original figures:", len(figure_index))
print("Figure fingerprints: PASS")

# ----------------------------------------------------------------------
# 3. Verify research evidence consistency
# ----------------------------------------------------------------------

models = pd.read_csv(
    output_dir / "NSL12_final_model_results.csv"
)

findings = pd.read_csv(
    output_dir / "NSL12_research_findings.csv"
)

mapping = pd.read_csv(
    output_dir
    / "NSL12_research_question_objectives_mapping.csv"
)

assert len(models) == 4
assert len(findings) == 10
assert len(mapping) == 4

assert set(models["Model"]) == {
    "RF-B-CIC",
    "RF-E-CIC",
    "RF-B-NSL",
    "RF-H2-NSL"
}

assert set(findings["Finding ID"]) == {
    f"F{i:02d}" for i in range(1, 11)
}

conclusion = NSL12_RESEARCH_CONCLUSION

assert conclusion["nsl_test_only_attack_records"] == 3750
assert conclusion["nsl_test_only_attack_errors"] == 3051

test_only_error_share = 3051 / 5447

assert abs(
    test_only_error_share - 0.56012
) < 0.00001

print("\nRESEARCH EVIDENCE")
print("Model references:", len(models))
print("Research findings:", len(findings))
print("Research mappings:", len(mapping))
print("Attack-novelty totals: PASS")
print("Evidence reconciliation: PASS")

# ----------------------------------------------------------------------
# 4. Create final integrity record
# ----------------------------------------------------------------------

NSL12_FINAL_INTEGRITY = {
    "notebook": "12_Final_Research_Evidence_Consolidation",
    "stage": "CELL_10",
    "status": "PASS",
    "previous_evidence_files_verified": 15,
    "export_audit_verified": True,
    "original_figures_verified": 5,
    "model_records": 4,
    "research_findings": 10,
    "research_mappings": 4,
    "evidence_fingerprint_status": "PASS",
    "research_reconciliation": "PASS",
    "original_source_integrity": (
        "Previously verified in Notebook 12 Cell 01"
    ),
    "model_training_performed": False,
    "source_experiments_modified_by_this_cell": False
}

integrity_path = (
    output_dir / "NSL12_final_integrity_record.json"
)

if integrity_path.exists():

    with open(integrity_path, "r", encoding="utf-8") as f:
        existing = json.load(f)

    assert existing == NSL12_FINAL_INTEGRITY
    integrity_status = "VERIFIED EXISTING"

else:

    with open(integrity_path, "w", encoding="utf-8") as f:
        json.dump(
            NSL12_FINAL_INTEGRITY,
            f,
            indent=2,
            ensure_ascii=False
        )

    integrity_status = "CREATED"

NSL12_CELL10_FILES = [integrity_path]

print("\nFINAL INTEGRITY RECORD")
print("Integrity record:", integrity_status)
print("Previous experiment integrity: VERIFIED IN CELL 01")
print("Current evidence fingerprints: PASS")
print("Research evidence reconciliation: PASS")
print("Model training performed: NO")

print("\nCELL 10: PASSED")
print("Final research-evidence integrity: VERIFIED")
print("=" * 72)

CELL 10 — FINAL RESEARCH-EVIDENCE INTEGRITY

PRIOR EVIDENCE AUDIT
Evidence files reverified: 15
Export audit record: PASS
SHA-256 verification: PASS

RESEARCH FIGURES
Verified original figures: 5
Figure fingerprints: PASS

RESEARCH EVIDENCE
Model references: 4
Research findings: 10
Research mappings: 4
Attack-novelty totals: PASS
Evidence reconciliation: PASS

FINAL INTEGRITY RECORD
Integrity record: CREATED
Previous experiment integrity: VERIFIED IN CELL 01
Current evidence fingerprints: PASS
Research evidence reconciliation: PASS
Model training performed: NO

CELL 10: PASSED
Final research-evidence integrity: VERIFIED


In [11]:
# ======================================================================
# CELL 11 — FINAL REPRODUCIBILITY MANIFEST
# ======================================================================

from pathlib import Path
import json
import pandas as pd

print("=" * 72)
print("CELL 11 — FINAL REPRODUCIBILITY MANIFEST")
print("=" * 72)

required = [
    "NSL12_OUTPUT_DIR",
    "NSL12_MODEL_PATHS",
    "NSL12_CELL04_FILES",
    "NSL12_CELL05_FILES",
    "NSL12_CELL06_FILES",
    "NSL12_CELL07_FILES",
    "NSL12_CELL08_FILES",
    "NSL12_CELL09_FILES",
    "NSL12_CELL10_FILES",
    "ns12_file_sha256"
]

missing = [name for name in required if name not in globals()]

if missing:
    raise RuntimeError(
        "Missing prerequisites: " + ", ".join(missing)
    )

output_dir = Path(NSL12_OUTPUT_DIR).resolve()

# ----------------------------------------------------------------------
# 1. Collect completed Notebook 12 exports
# ----------------------------------------------------------------------

previous_files = [
    Path(p).resolve()
    for p in (
        NSL12_CELL04_FILES
        + NSL12_CELL05_FILES
        + NSL12_CELL06_FILES
        + NSL12_CELL07_FILES
        + NSL12_CELL08_FILES
        + NSL12_CELL09_FILES
        + NSL12_CELL10_FILES
    )
]

assert len(previous_files) == 17
assert len(set(previous_files)) == 17

file_records = []

for path in sorted(previous_files, key=lambda p: p.name):

    assert path.is_file()
    assert path.parent == output_dir

    file_records.append({
        "filename": path.name,
        "size_bytes": path.stat().st_size,
        "sha256": ns12_file_sha256(path)
    })

# ----------------------------------------------------------------------
# 2. Preserve existing model references
# ----------------------------------------------------------------------

assert isinstance(NSL12_MODEL_PATHS, dict)
assert len(NSL12_MODEL_PATHS) == 4

model_references = {
    str(model_name): str(model_path)
    for model_name, model_path in NSL12_MODEL_PATHS.items()
}

# These reference already verified models.
# No loading, retraining or model export is performed here.

# ----------------------------------------------------------------------
# 3. Preserve archived figure references
# ----------------------------------------------------------------------

figure_index = pd.read_csv(
    output_dir / "NSL12_final_figure_index.csv"
)

figure_references = []

for _, row in figure_index.iterrows():

    figure_references.append({
        "figure_id": row["Figure ID"],
        "filename": row["Filename"],
        "relative_path": row["Relative Path"],
        "sha256": row["SHA256"]
    })

assert len(figure_references) == 5

# ----------------------------------------------------------------------
# 4. Build reproducibility manifest
# ----------------------------------------------------------------------

NSL12_FINAL_MANIFEST = {
    "notebook": "12_Final_Research_Evidence_Consolidation",

    "manifest_stage": "CELL_11",

    "status": "VERIFIED",

    "datasets": [
        "CIC-IDS2017",
        "NSL-KDD"
    ],

    "source_evidence": {
        "cic_statistical_files": 17,
        "nsl_replication_files": 22,
        "cross_benchmark_files": 18,
        "source_reference_files": 14,
        "verification": (
            "Verified in Notebook 12 Cell 01"
        )
    },

    "model_references": model_references,

    "notebook12_evidence_files": file_records,

    "figure_references": figure_references,

    "experimental_scope": {
        "independent_benchmark_replication": True,
        "direct_cic_model_transfer": False,
        "additional_training": False,
        "additional_hyperparameter_search": False,
        "additional_resampling": False,
        "additional_test_evaluation": False
    },

    "interpretation_controls": {
        "different_benchmark_label_spaces": True,
        "raw_macro_f1_ranking_excluded": True,
        "mcnemar_scope_limited_to_correctness": True,
        "single_cause_shift_claim_excluded": True
    }
}

# ----------------------------------------------------------------------
# 5. Save manifest without overwriting different content
# ----------------------------------------------------------------------

manifest_path = (
    output_dir / "NSL12_final_reproducibility_manifest.json"
)

if manifest_path.exists():

    with open(manifest_path, "r", encoding="utf-8") as f:
        existing_manifest = json.load(f)

    assert existing_manifest == NSL12_FINAL_MANIFEST
    manifest_status = "VERIFIED EXISTING"

else:

    with open(manifest_path, "w", encoding="utf-8") as f:
        json.dump(
            NSL12_FINAL_MANIFEST,
            f,
            indent=2,
            ensure_ascii=False
        )

    manifest_status = "CREATED"

with open(manifest_path, "r", encoding="utf-8") as f:
    persisted_manifest = json.load(f)

assert persisted_manifest == NSL12_FINAL_MANIFEST

NSL12_FINAL_MANIFEST_SHA256 = ns12_file_sha256(
    manifest_path
)

NSL12_CELL11_FILES = [manifest_path]

# ----------------------------------------------------------------------
# 6. Verification
# ----------------------------------------------------------------------

print("\nFINAL MANIFEST SUMMARY")
print("Manifest:", manifest_status)
print("Datasets:", ", ".join(NSL12_FINAL_MANIFEST["datasets"]))
print("Model references:", len(model_references))
print("Source reference files:", 14)
print("Notebook 12 fingerprinted evidence files:", len(file_records))
print("Original research figure references:", len(figure_references))

print("\nMANIFEST SHA-256")
print(NSL12_FINAL_MANIFEST_SHA256)

print("\nREPRODUCIBILITY CHECKS")
print("Evidence file fingerprints: RECORDED")
print("Figure references: RECORDED")
print("Persisted manifest verification: PASS")
print("Experimental configurations: PRESERVED")
print("Source experiments modified: NO")
print("Model training performed: NO")

print("\nCELL 11: PASSED")
print("Final reproducibility manifest: VERIFIED")
print("=" * 72)

CELL 11 — FINAL REPRODUCIBILITY MANIFEST

FINAL MANIFEST SUMMARY
Manifest: CREATED
Datasets: CIC-IDS2017, NSL-KDD
Model references: 4
Source reference files: 14
Notebook 12 fingerprinted evidence files: 17
Original research figure references: 5

MANIFEST SHA-256
fcd236437b6f896bfdd3b280fb67595bb5011482d5876e760f0656d14993be91

REPRODUCIBILITY CHECKS
Evidence file fingerprints: RECORDED
Figure references: RECORDED
Persisted manifest verification: PASS
Experimental configurations: PRESERVED
Source experiments modified: NO
Model training performed: NO

CELL 11: PASSED
Final reproducibility manifest: VERIFIED


In [12]:
# ======================================================================
# CELL 12 — FINAL NOTEBOOK COMPLETION RECORD
# ======================================================================

from pathlib import Path
import json
import pandas as pd

print("=" * 72)
print("CELL 12 — FINAL NOTEBOOK COMPLETION RECORD")
print("=" * 72)

required = [
    "PROJECT_ROOT",
    "NSL12_OUTPUT_DIR",
    "NSL12_CELL04_FILES",
    "NSL12_CELL05_FILES",
    "NSL12_CELL06_FILES",
    "NSL12_CELL07_FILES",
    "NSL12_CELL08_FILES",
    "NSL12_CELL09_FILES",
    "NSL12_CELL10_FILES",
    "NSL12_CELL11_FILES",
    "ns12_file_sha256"
]

missing = [name for name in required if name not in globals()]

if missing:
    raise RuntimeError(
        "Missing prerequisites: " + ", ".join(missing)
    )

output_dir = Path(NSL12_OUTPUT_DIR).resolve()

# ----------------------------------------------------------------------
# 1. Reload final manifest
# ----------------------------------------------------------------------

manifest_path = Path(NSL12_CELL11_FILES[0]).resolve()

assert manifest_path.is_file()

with open(manifest_path, "r", encoding="utf-8") as f:
    final_manifest = json.load(f)

assert final_manifest["status"] == "VERIFIED"

manifest_sha256 = ns12_file_sha256(manifest_path)

# ----------------------------------------------------------------------
# 2. Reverify all manifest evidence files
# ----------------------------------------------------------------------

manifest_files = final_manifest["notebook12_evidence_files"]

assert len(manifest_files) == 17

for record in manifest_files:

    path = (
        output_dir / record["filename"]
    ).resolve()

    assert path.parent == output_dir
    assert path.is_file()

    assert path.stat().st_size == record["size_bytes"]

    assert ns12_file_sha256(path) == record["sha256"]

print("\nFINAL EVIDENCE VERIFICATION")
print("Manifest-referenced files:", len(manifest_files))
print("File sizes: PASS")
print("Evidence SHA-256 fingerprints: PASS")

# ----------------------------------------------------------------------
# 3. Reverify original research figure references
# ----------------------------------------------------------------------

figure_references = final_manifest["figure_references"]

assert len(figure_references) == 5

for record in figure_references:

    path = (
        Path(PROJECT_ROOT) / record["relative_path"]
    ).resolve()

    assert path.is_file()

    assert ns12_file_sha256(path) == record["sha256"]

print("\nORIGINAL FIGURE VERIFICATION")
print("Referenced figures:", len(figure_references))
print("Figure SHA-256 fingerprints: PASS")

# ----------------------------------------------------------------------
# 4. Verify all completed Notebook 12 outputs
# ----------------------------------------------------------------------

all_previous_outputs = [
    Path(p).resolve()
    for p in (
        NSL12_CELL04_FILES
        + NSL12_CELL05_FILES
        + NSL12_CELL06_FILES
        + NSL12_CELL07_FILES
        + NSL12_CELL08_FILES
        + NSL12_CELL09_FILES
        + NSL12_CELL10_FILES
        + NSL12_CELL11_FILES
    )
]

assert len(all_previous_outputs) == 18
assert len(set(all_previous_outputs)) == 18
assert all(path.is_file() for path in all_previous_outputs)

# ----------------------------------------------------------------------
# 5. Create completion record
# ----------------------------------------------------------------------

NSL12_COMPLETION_RECORD = {
    "notebook": "12_Final_Research_Evidence_Consolidation",
    "status": "COMPLETE",
    "completed_cells": 12,

    "datasets": [
        "CIC-IDS2017",
        "NSL-KDD"
    ],

    "model_references": 4,
    "research_findings": 10,
    "research_figures": 5,
    "research_question_mappings": 4,

    "manifest_evidence_files": 17,
    "files_before_completion_record": 18,
    "total_notebook12_output_files": 19,

    "reproducibility_manifest": manifest_path.name,
    "reproducibility_manifest_sha256": manifest_sha256,

    "evidence_integrity": "PASS",
    "figure_integrity": "PASS",
    "manifest_integrity": "PASS",

    "model_training_performed": False,
    "source_experiments_modified": False,

    "next_stage": (
        "Dissertation Results, Discussion and Final Write-up"
    )
}

completion_path = (
    output_dir / "NSL12_completion_record.json"
)

if completion_path.exists():

    with open(completion_path, "r", encoding="utf-8") as f:
        previous_completion = json.load(f)

    assert previous_completion == NSL12_COMPLETION_RECORD

    completion_status = "VERIFIED EXISTING"

else:

    with open(completion_path, "w", encoding="utf-8") as f:
        json.dump(
            NSL12_COMPLETION_RECORD,
            f,
            indent=2,
            ensure_ascii=False
        )

    completion_status = "CREATED"

# ----------------------------------------------------------------------
# 6. Independently reload and verify completion
# ----------------------------------------------------------------------

with open(completion_path, "r", encoding="utf-8") as f:
    saved_completion = json.load(f)

assert saved_completion == NSL12_COMPLETION_RECORD

completion_sha256 = ns12_file_sha256(completion_path)

NSL12_CELL12_FILES = [completion_path]

# ----------------------------------------------------------------------
# 7. Final completion audit
# ----------------------------------------------------------------------

print("\nFINAL NOTEBOOK 12 AUDIT")
print("Source evidence integrity: PASS")
print("Model-result consolidation: PASS")
print("Research findings: PASS")
print("Dissertation tables: PASS")
print("Research figures: PASS")
print("Research objective mapping: PASS")
print("Export integrity: PASS")
print("Final reproducibility manifest: PASS")
print("Completion record verification: PASS")
print("Model training performed: NO")
print("Source experiments modified: NO")

print("\nCOMPLETION RECORD")
print("Completion record:", completion_status)
print("Notebook status: COMPLETE")
print("Completed cells:", saved_completion["completed_cells"])
print("Model references:", saved_completion["model_references"])
print("Research figures:", saved_completion["research_figures"])
print(
    "Final Notebook 12 output files:",
    saved_completion["total_notebook12_output_files"]
)

print("\nREPRODUCIBILITY MANIFEST SHA-256")
print(manifest_sha256)

print("\nCOMPLETION RECORD SHA-256")
print(completion_sha256)

print("\n" + "=" * 72)
print("NOTEBOOK 12: COMPLETE")
print("Final evidence package: VERIFIED")
print("Next stage: Dissertation Results, Discussion and Final Write-up")
print("=" * 72)

CELL 12 — FINAL NOTEBOOK COMPLETION RECORD

FINAL EVIDENCE VERIFICATION
Manifest-referenced files: 17
File sizes: PASS
Evidence SHA-256 fingerprints: PASS

ORIGINAL FIGURE VERIFICATION
Referenced figures: 5
Figure SHA-256 fingerprints: PASS

FINAL NOTEBOOK 12 AUDIT
Source evidence integrity: PASS
Model-result consolidation: PASS
Research findings: PASS
Dissertation tables: PASS
Research figures: PASS
Research objective mapping: PASS
Export integrity: PASS
Final reproducibility manifest: PASS
Completion record verification: PASS
Model training performed: NO
Source experiments modified: NO

COMPLETION RECORD
Completion record: CREATED
Notebook status: COMPLETE
Completed cells: 12
Model references: 4
Research figures: 5
Final Notebook 12 output files: 19

REPRODUCIBILITY MANIFEST SHA-256
fcd236437b6f896bfdd3b280fb67595bb5011482d5876e760f0656d14993be91

COMPLETION RECORD SHA-256
318410e22c9bee2e10cd2e98bcc996a874d9ed5c98605a59b487e3df235bfbf3

NOTEBOOK 12: COMPLETE
Final evidence package: 